In [1]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [2]:
df = pd.read_csv(r'/home/user/Downloads/DataHut_AT_Dm_PriceExtractions_20261103.CSV', low_memory=False, sep='|')

In [3]:
df.shape

(17217, 21)

In [5]:
import pandas as pd
import re



print("Total records:", len(df))
print("Total columns:", len(df.columns))

issues = []

# Check text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns

for col in text_cols:

    s = df[col].astype(str)

    # Leading/trailing whitespace
    mask = s.str.contains(r"^\s|\s$", regex=True)

    # Multiple consecutive spaces
    mask |= s.str.contains(r" {2,}", regex=True)

    # Tab characters
    mask |= s.str.contains(r"\t", regex=True)

    # Newline / carriage return
    mask |= s.str.contains(r"[\r\n]", regex=True)

    # Unescaped/control characters
    mask |= s.str.contains(
        r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]",
        regex=True
    )

    if mask.any():

        # Only the FIRST issue from this column
        idx = df.index[mask][0]

        issues.append({
            "row_no": idx + 2,
            "url": df.loc[idx, "pdp_url"],
            "column": col
        })


# Display results
if issues:

    result = pd.DataFrame(issues)

    print("\n==============================")
    print("Total columns with issues:", len(result))
    print("==============================\n")

    print(result.to_string(index=False))

else:

    print("\nNo whitespace, extra space, newline, tab, or unescaped character issues found.")

Total records: 43331
Total columns: 21

Total columns with issues: 1

 row_no                                                                                                           url     column
  26686 https://www.spar.at/produktwelt/vorarlberg-milch-laendle-halbfettmilch-laenger-frisch-1-5-fett-p2020006312547 breadcrumb


In [6]:
import pandas as pd

# Promotion date columns
date_cols = ["promotion_valid_from", "promotion_valid_upto"]

# Convert dates without modifying original data
dates = df[date_cols].apply(
    lambda col: pd.to_datetime(col, errors="coerce", format="mixed")
)

today = pd.Timestamp.today().normalize()

# 1. Invalid date formats (excluding empty values)
for col in date_cols:
    original = df[col].astype("string").str.strip()

    mask = (
        original.notna()
        & original.ne("")
        & dates[col].isna()
    )

    result = df.loc[mask, ["pdp_url", col]].copy()
    result.insert(0, "row_no", result.index + 2)

    print(f"\nInvalid dates in {col}: {len(result)}")
    print(result.head(20).to_string(index=False))


# 2. Future promotion start dates
mask = dates["promotion_valid_from"] > today

result = df.loc[
    mask, ["pdp_url", "promotion_valid_from", "promotion_valid_upto"]
].copy()

result.insert(0, "row_no", result.index + 2)

print("\nFuture promotion start dates:", len(result))
print(result.head(20).to_string(index=False))


# 3. Future promotion end dates
mask = dates["promotion_valid_upto"] > today

result = df.loc[
    mask, ["pdp_url", "promotion_valid_from", "promotion_valid_upto"]
].copy()

result.insert(0, "row_no", result.index + 2)

print("\nFuture promotion end dates:", len(result))
print(result.head(20).to_string(index=False))


# 4. Promotion start date greater than end date
mask = (
    dates["promotion_valid_from"].notna()
    & dates["promotion_valid_upto"].notna()
    & (dates["promotion_valid_from"] > dates["promotion_valid_upto"])
)

result = df.loc[
    mask, ["pdp_url", "promotion_valid_from", "promotion_valid_upto"]
].copy()

result.insert(0, "row_no", result.index + 2)

print("\nInvalid promotion date ranges:", len(result))
print(result.head(20).to_string(index=False))


# 5. Overall promotion date ranges
print("\nPromotion date ranges:")

for col in date_cols:
    print(
        f"{col}:",
        dates[col].min(),
        "to",
        dates[col].max()
    )


Invalid dates in promotion_valid_from: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_from]
Index: []

Invalid dates in promotion_valid_upto: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_upto]
Index: []

Future promotion start dates: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_from, promotion_valid_upto]
Index: []

Future promotion end dates: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_from, promotion_valid_upto]
Index: []

Invalid promotion date ranges: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_from, promotion_valid_upto]
Index: []

Promotion date ranges:
promotion_valid_from: NaT to NaT
promotion_valid_upto: NaT to NaT


In [7]:
import pandas as pd


# Check missing or unwanted IDs
unwanted_values = ["NULL", "NAN", "NONE", "N/A", "NA", "UNDEFINED"]

id_text = df["unique_id"].astype("string").str.strip()

mask = (
    id_text.isna()
    | id_text.eq("")
    | id_text.str.upper().isin(unwanted_values)
)

result = df.loc[mask, ["unique_id", "pdp_url"]].copy()
result["row_no"] = result.index + 2

print("Missing/unwanted IDs:", len(result))
print(result[["row_no", "unique_id", "pdp_url"]].to_string(index=False))

# Check duplicate IDs
duplicate_mask = (
    id_text.notna()
    & id_text.ne("")
    & ~id_text.str.upper().isin(unwanted_values)
    & id_text.duplicated(keep=False)
)

duplicates = df.loc[duplicate_mask, ["unique_id", "pdp_url"]].copy()
duplicates["row_no"] = duplicates.index + 2

print("\nDuplicate ID records:", len(duplicates))
print(
    duplicates[["row_no", "unique_id", "pdp_url"]]
    .sort_values("unique_id")
    .to_string(index=False)
)

Missing/unwanted IDs: 0
Empty DataFrame
Columns: [row_no, unique_id, pdp_url]
Index: []

Duplicate ID records: 0
Empty DataFrame
Columns: [row_no, unique_id, pdp_url]
Index: []


In [8]:
import pandas as pd

value = df["unique_id"].fillna("").astype(str).str.strip()

mask = value.ne("") & ~value.str.fullmatch(r"\d+", na=False)

result = df.loc[mask, ["pdp_url", "unique_id"]].copy()
result["record_no"] = result.index + 2
result["field"] = "unique_id"

print(result[["record_no", "pdp_url", "field", "unique_id"]].to_string(index=False))

Empty DataFrame
Columns: [record_no, pdp_url, field, unique_id]
Index: []


In [9]:
req_cols = """unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,
site_shown_uom



"""

req_cols = [c.strip() for c in req_cols.split(",") if c.strip()]

missing = [c for c in req_cols if c not in df.columns]
extra = [c for c in df.columns if c not in req_cols]

print("Requirement columns" 
":", len(req_cols))
print("File columns:", len(df.columns))
print("Missing:", missing)
print("Extra:", extra)
print("Order matches:", req_cols == list(df.columns))

Requirement columns: 21
File columns: 21
Missing: []
Extra: []
Order matches: True


In [10]:
# Empty columns as per requirement
req_empty = """price_valid_from,percentage_discount,promotion_valid_from,promotion_valid_upto,region,pack_size"""

req_empty = [c.strip() for c in req_empty.split(",") if c.strip()]

# Empty columns in the file
file_empty = df.columns[df.isna().all()].tolist()

# Compare
missing_empty = [c for c in req_empty if c not in file_empty]
unexpected_empty = [c for c in file_empty if c not in req_empty]
matching_empty = [c for c in req_empty if c in file_empty]

print("Requirement empty columns :", len(req_empty))
print("File empty columns        :", len(file_empty))
print("Matching                 :", len(matching_empty))
print("Expected empty but not empty:", missing_empty)
print("Unexpected empty columns     :", unexpected_empty)

Requirement empty columns : 6
File empty columns        : 6
Matching                 : 6
Expected empty but not empty: []
Unexpected empty columns     : []


In [11]:
# Check invalid price values
# Digits, comma and decimal point are allowed

price = df["price"].fillna("").astype(str).str.strip()

mask = (
    price.ne("")
    & ~price.str.fullmatch(r"\d+(,\d+)*(\.\d+)?", na=False)
)

result = df.loc[mask, ["url", "price"]].copy()
result["record_no"] = result.index + 2

print("Invalid price values:", len(result))
print(
    result[
        ["record_no", "url", "price"]
    ].to_string(index=False)
)

KeyError: 'price'

In [11]:
df[df.currency.isna()].price.value_counts()

AttributeError: 'DataFrame' object has no attribute 'price'

In [12]:
df.currency.value_counts()

currency
EUR    43331
Name: count, dtype: int64

In [13]:
df.competitor_name.value_counts()

competitor_name
spar    43331
Name: count, dtype: int64

In [16]:
df.instock.value_counts()

AttributeError: 'DataFrame' object has no attribute 'instock'

In [17]:
df.organictype.value_counts()

AttributeError: 'DataFrame' object has no attribute 'organictype'

In [14]:
df.duplicated().sum()

np.int64(0)

In [15]:
df.pdp_url.duplicated().sum()

np.int64(0)

In [16]:
df.unique_id.duplicated().sum()

np.int64(0)

In [21]:
import re

value = df["company_phone"].fillna("").astype(str).str.strip()

digits = value.str.replace(r"\D", "", regex=True)

mask = (
    value.ne("")
    & (
        ~value.str.match(r"^[0-9+().\-\s]+$", na=False)
        | (digits.str.len() < 7)
    )
)

result = df.loc[mask, ["url", "company_phone"]].copy()
result["record_no"] = result.index + 2
result["field"] = "company_phone"

print(result[["record_no", "url", "field", "company_phone"]].to_string(index=False))

KeyError: 'company_phone'

In [30]:
text_cols = [
    "main_category",
    "category",
    "sub_category",
    "company_name",
    "company_location",
    "contact_person"
]

for col in text_cols:

    value = df[col].fillna("").astype(str).str.strip()

    mask = value.ne("") & value.str.fullmatch(r"\d+", na=False)

    result = df.loc[mask, ["url", col]].copy()
    result["record_no"] = result.index + 2
    result["field"] = col

    print(f"\nInvalid {col}: {len(result)}")
    print(
        result[["record_no", "url", "field", col]]
        .head(5)
        .to_string(index=False)
    )

KeyError: 'main_category'

In [31]:
value = df["spec_fields"].fillna("").astype(str).str.strip()

mask = value.str.contains(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", regex=True, na=False)

result = df.loc[mask, ["url", "spec_fields"]].copy()
result["record_no"] = result.index + 2
result["field"] = "spec_fields"

print(result[["record_no", "url", "field", "spec_fields"]].head(5).to_string(index=False))

KeyError: 'spec_fields'

In [17]:
df.extraction_date.value_counts()

extraction_date
2026-11-03    43331
Name: count, dtype: int64

In [18]:
df.isnull().sum()

unique_id                    0
competitor_name              0
extraction_date              0
product_name                 0
grammage_quantity            0
grammage_unit                0
regular_price               41
selling_price               41
price_valid_from         43331
price_per_unit           17821
percentage_discount      43331
promotion_price          41436
promotion_valid_from     43331
promotion_valid_upto     43331
promotion_description    41436
currency                     0
breadcrumb                   0
pdp_url                      0
region                   43331
pack_size                43331
site_shown_uom               0
dtype: int64

In [72]:
df[df.product_description.isna()].pdp_url.head()

AttributeError: 'DataFrame' object has no attribute 'product_description'

In [19]:
import pandas as pd

# ============================================================
# 1. CHECK INVALID unique_id VALUES
# ============================================================

unique_id = df["unique_id"].fillna("").astype(str).str.strip()

# Invalid if anything other than digits is present
invalid_mask = ~unique_id.str.fullmatch(r"\d+")

invalid_result = df.loc[
    invalid_mask,
    ["unique_id", "pdp_url"]
].copy()

invalid_result["row_no"] = invalid_result.index + 2

invalid_result = invalid_result[
    ["row_no", "pdp_url", "unique_id"]
]


# ============================================================
# 2. CHECK DUPLICATE unique_id VALUES
# ============================================================

duplicate_mask = df["unique_id"].duplicated(keep=False)

duplicate_result = df.loc[
    duplicate_mask,
    ["unique_id", "pdp_url"]
].copy()

duplicate_result["row_no"] = duplicate_result.index + 2

duplicate_result = duplicate_result[
    ["row_no", "pdp_url", "unique_id"]
]


# ============================================================
# 3. PRINT INVALID IDs
# ============================================================

print("\n==============================")
print("Invalid unique_id")
print("==============================")

print("Total invalid unique_id:", len(invalid_result))

if len(invalid_result) > 0:
    print(invalid_result.to_string(index=False))
else:
    print("No invalid unique_id found.")


# ============================================================
# 4. PRINT DUPLICATE IDs
# ============================================================

print("\n==============================")
print("Duplicate unique_id")
print("==============================")

print("Total duplicate records:", len(duplicate_result))

if len(duplicate_result) > 0:
    print(duplicate_result.to_string(index=False))
else:
    print("No duplicate unique_id found.")


# ============================================================
# 5. SUMMARY
# ============================================================

print("\n==============================")
print("Summary")
print("==============================")

print("Total records:", len(df))
print("Unique unique_id count:", df["unique_id"].nunique())
print("Invalid unique_id:", len(invalid_result))
print("Duplicate records:", len(duplicate_result))


Invalid unique_id
Total invalid unique_id: 0
No invalid unique_id found.

Duplicate unique_id
Total duplicate records: 0
No duplicate unique_id found.

Summary
Total records: 43331
Unique unique_id count: 43331
Invalid unique_id: 0
Duplicate records: 0


In [21]:
import pandas as pd

hierarchy_cols = [
    "producthierarchy_level1",
    "producthierarchy_level2",
    "producthierarchy_level3",
    "producthierarchy_level4",
    "producthierarchy_level5",
    "producthierarchy_level6",
    "producthierarchy_level7"
]

issues = []

for idx, row in df.iterrows():

    breadcrumb = str(row["breadcrumb"]).strip()

    if not breadcrumb:
        continue

    # Split breadcrumb using >
    breadcrumb_levels = [
        x.strip()
        for x in breadcrumb.split(">")
    ]

    for i, breadcrumb_value in enumerate(breadcrumb_levels):

        if i >= len(hierarchy_cols):
            break

        hierarchy_col = hierarchy_cols[i]
        hierarchy_value = row[hierarchy_col]

        # Breadcrumb has value but hierarchy is missing
        if (
            breadcrumb_value != ""
            and (
                pd.isna(hierarchy_value)
                or str(hierarchy_value).strip() == ""
            )
        ):
            issues.append({
                "row_no": idx + 2,
                "pdp_url": row["pdp_url"],
                "breadcrumb": breadcrumb,
                "missing_hierarchy": hierarchy_col,
                "breadcrumb_value": breadcrumb_value
            })

result = pd.DataFrame(issues)

print("Total issues:", len(result))

if not result.empty:
    print(result.to_string(index=False))

KeyError: 'producthierarchy_level1'

In [20]:
import pandas as pd

# Convert price fields to numeric
promotion_price = pd.to_numeric(
    df["promotion_price"], errors="coerce"
)

multibuy_total = pd.to_numeric(
    df["multi_buy_items_price_total"], errors="coerce"
)

selling_price = pd.to_numeric(
    df["selling_price"], errors="coerce"
)

# Normalize promotion_type
promotion_type = (
    df["promotion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Expected promotion_price:
# multibuy → multi_buy_items_price_total
# otherwise → selling_price
expected_price = multibuy_total.where(
    promotion_type == "multibuy",
    selling_price
)

# Find where promotion_price is different from expected value
mask = (
    promotion_price.notna() &
    expected_price.notna() &
    (promotion_price != expected_price)
)

issues = df.loc[
    mask,
    [
        "promotion_type",
        "promotion_price",
        "multi_buy_items_price_total",
        "selling_price",
        "pdp_url"
    ]
].copy()

# Add Excel row number
issues["row_no"] = issues.index + 2

# Arrange columns
issues = issues[
    [
        "row_no",
        "pdp_url",
        "promotion_type",
        "promotion_price",
        "multi_buy_items_price_total",
        "selling_price"
    ]
]

print(f"Total promotion_price issues: {len(issues)}")
print(issues.to_string(index=False))

KeyError: 'multi_buy_items_price_total'

In [21]:
import pandas as pd

issues = []

# -------------------------------
# Normalize columns
# -------------------------------

df["promotion_description_clean"] = (
    df["promotion_description"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["producthierarchy_level3_clean"] = (
    df["producthierarchy_level3"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["producthierarchy_level4_clean"] = (
    df["producthierarchy_level4"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Convert prices
for col in [
    "regular_price",
    "selling_price",
    "promotion_price",
    "price_was"
]:
    df[col + "_num"] = pd.to_numeric(df[col], errors="coerce")


# ==================================================
# 1. Fleisch & Fisch validation
# ==================================================

fleisch_mask = (
    (df["producthierarchy_level3_clean"] == "fleisch & fisch") &
    (df["producthierarchy_level4_clean"] != "charcuterie & wurstwaren")
)

fleisch = df[fleisch_mask]


# price_was should be empty
mask = (
    fleisch["price_was"]
    .astype("string")
    .fillna("")
    .str.strip()
    .ne("")
)

for idx in fleisch[mask].index:
    issues.append([
        idx + 2,
        df.loc[idx, "pdp_url"],
        "Fleisch & Fisch issue",
        "price_was should be empty but value is present"
    ])


# ==================================================
# 2. HIT Promotion validation
# ==================================================

hit = df[
    df["promotion_description_clean"] == "hit"
]


for idx, row in hit.iterrows():

    regular = row["regular_price_num"]
    selling = row["selling_price_num"]
    promotion = row["promotion_price_num"]

    price_was = row["price_was_num"]

    # Single price HIT
    if pd.notna(regular) and pd.notna(selling) and regular == selling:

        if (
            promotion != selling or
            str(row["regular_price"]).strip() not in ["", "nan"] or
            str(row["price_was"]).strip() not in ["", "nan"]
        ):
            issues.append([
                idx + 2,
                row["pdp_url"],
                "HIT promotion issue",
                "Single price HIT: promotion_price should equal selling_price and regular_price/price_was should be empty"
            ])

    # Two price HIT
    elif pd.notna(regular) and pd.notna(selling) and regular != selling:

        if (
            promotion != selling or
            price_was != regular
        ):
            issues.append([
                idx + 2,
                row["pdp_url"],
                "HIT promotion issue",
                "Two price HIT: promotion_price should equal selling_price and price_was should equal regular_price"
            ])


# ==================================================
# Final output
# ==================================================

if issues:
    result = pd.DataFrame(
        issues,
        columns=[
            "row_no",
            "pdp_url",
            "issue_type",
            "issue_description"
        ]
    )

    print(f"Total issues found: {len(result)}")
    print(result.to_string(index=False))

else:
    print("No issues found.")

KeyError: 'producthierarchy_level3'

In [22]:
import pandas as pd

promotion_price = pd.to_numeric(df["promotion_price"], errors="coerce")
selling_price = pd.to_numeric(df["selling_price"], errors="coerce")
multibuy_total = pd.to_numeric(
    df["multi_buy_items_price_total"], errors="coerce"
)

promotion_type = (
    df["promotion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Expected promotion price
expected_price = multibuy_total.where(
    promotion_type == "multibuy",
    selling_price
)

# Find mismatches
mask = (
    promotion_price.notna() &
    expected_price.notna() &
    (promotion_price != expected_price)
)

issues = df.loc[
    mask,
    [
        "promotion_type",
        "promotion_price",
        "selling_price",
        "multi_buy_items_price_total",
        "pdp_url"
    ]
].copy()

# Row number
issues["row_no"] = issues.index + 2

issues = issues[
    [
        "row_no",
        "pdp_url",
        "promotion_type",
        "promotion_price",
        "selling_price",
        "multi_buy_items_price_total"
    ]
]

print(f"Total promotion price issues: {len(issues)}")
print(issues.to_string(index=False))

KeyError: 'multi_buy_items_price_total'

In [23]:
import pandas as pd

desc = (
    df["promotion_description"]
    .astype("string")
    .str.strip()
    .str.lower()
)

promo_type = (
    df["promotion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Expected promotion type
expected_type = pd.Series(pd.NA, index=df.index, dtype="string")


# MultiBuy: description contains "for"
expected_type.loc[desc.str.contains(r"\bfor\b", na=False)] = "multibuy"

# % Discount: description contains was / save / % off / 1/2 price
discount_mask = desc.str.contains(
    r"\bwas\b|\bsave\b|%[\s]*off|\b1/2\s*price\b",
    regex=True,
    na=False
)

expected_type.loc[discount_mask] = "% discount"

# Find mismatches
mask = (
    expected_type.notna() &
    promo_type.notna() &
    (promo_type != expected_type)
)

issues = df.loc[
    mask,
    ["pdp_url", "promotion_description", "promotion_type"]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    ["row_no", "pdp_url", "promotion_description", "promotion_type"]
]

print(f"Total promotion_type issues: {len(issues)}")
print(issues.to_string(index=False))

KeyError: 'promotion_type'

In [24]:
import pandas as pd

# Normalize promotion type
promotion_type = (
    df["promotion_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Convert prices to numeric
promotion_price = pd.to_numeric(df["promotion_price"], errors="coerce")
selling_price = pd.to_numeric(df["selling_price"], errors="coerce")

# Find issues
mask = (
    promotion_type.notna() &
    (promotion_type != "multibuy") &
    promotion_price.notna() &
    selling_price.notna() &
    (promotion_price != selling_price)
)

issues = df.loc[
    mask,
    [
        "pdp_url",
        "promotion_type",
        "promotion_price",
        "selling_price"
    ]
].copy()

# Add row number
issues["row_no"] = issues.index + 2

issues = issues[
    [
        "row_no",
        "pdp_url",
        "promotion_type",
        "promotion_price",
        "selling_price"
    ]
]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

KeyError: 'promotion_type'

In [25]:
# Check: promotion information is present but promotion_price is missing

percentage_discount = df['percentage_discount'].fillna('').astype(str).str.strip()
promotion_description = df['promotion_description'].fillna('').astype(str).str.strip()
promotion_price = df['promotion_price'].fillna('').astype(str).str.strip()

mask = (
    (percentage_discount != '') &
    (promotion_description != '') &
    (promotion_price == '')
)

issues = df.loc[
    mask,
    ['pdp_url', 'percentage_discount', 'promotion_description', 'promotion_price']
].copy()

# Excel row number
issues['row_no'] = issues.index + 2

issues = issues[
    ['row_no', 'pdp_url', 'percentage_discount',
     'promotion_description', 'promotion_price']
]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, percentage_discount, promotion_description, promotion_price]
Index: []


In [35]:
mask = (
    df["product_unique_key"].astype(str)
    !=
    (df["unique_id"].astype(str) + "P")
)

issues = df.loc[mask, ["unique_id", "product_unique_key", "pdp_url"]].copy()

issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "unique_id", "product_unique_key"]]

issues

KeyError: 'product_unique_key'

In [27]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

issues = {}

for col in text_cols:
    mask = (
        df[col]
        .fillna("")
        .astype(str)
        .str.contains(r"(?<!\\)\|", regex=True)
    )

    if mask.any():
        temp = df.loc[mask, ["pdp_url", col]].copy()
        temp.insert(0, "row_no", temp.index)  # DataFrame row number
        issues[col] = temp

# Print results
if issues:
    print("Columns containing unescaped '|':\n")

    for col, data in issues.items():
        print(f"\n=== {col} ({len(data)} rows) ===")
        print(data.to_string(index=False))
else:
    print("No unescaped '|' found.")

Columns containing unescaped '|':


=== product_name (10 rows) ===
 row_no                                                                                          pdp_url                                         product_name
   7671              https://www.spar.at/produktwelt/isana-cleanser-aha-pha-peeling-125ml-p2020006024686                      Isana AHA | PHA Cleanser 125 ml
  16941                    https://www.spar.at/produktwelt/mehrzweckschrauben-4x40-150stk-p2020005763197     BGS Mehrzweckschrauben | 4,0 x 40 mm | 150 Stück
  18049     https://www.spar.at/produktwelt/bgs-mehrzweckschrauben-4-0-x-50-mm-120-stueck-p2020005763210     BGS Mehrzweckschrauben | 4,0 x 50 mm | 120 Stück
  18199 https://www.spar.at/produktwelt/bgs-mehrzweckschrauben-t20-4-0-x-40-mm-150-stueck-p2020005762794 BGS Mehrzweckschrauben T20 | 4,0 x 40 mm | 150 Stück
  18952  https://www.spar.at/produktwelt/bgs-mehrzweckschrauben-t25-5-0-x-80-mm-50-stueck-p2020005763043  BGS Mehrzweckschrauben T25 | 5,0 x 80

In [28]:
import pandas as pd

escape_pattern = r'[\n\r\t\b\f\v]'

issues = []

for col in df.select_dtypes(include="object").columns:
    mask = df[col].fillna("").astype(str).str.contains(
        escape_pattern,
        regex=True,
        na=False
    )

    if mask.any():
        temp = df.loc[mask, [col]].copy()
        temp.insert(0, "row_no", temp.index + 2)
        temp["column"] = col
        temp["issue_value"] = temp[col]

        if "pdp_url" in df.columns:
            temp["pdp_url"] = df.loc[mask, "pdp_url"]

        issues.append(
            temp[["row_no", "pdp_url", "column", "issue_value"]]
        )

if issues:
    result = pd.concat(issues, ignore_index=True)

    print("=== Escape Character Issues ===")
    print(result.to_string(index=False))
    print(f"\nTotal issues: {len(result)}")
else:
    print("No escape characters found.")

No escape characters found.


In [29]:
df.nunique()

unique_id                      43331
competitor_name                    1
extraction_date                    1
product_name                   40654
grammage_quantity                665
grammage_unit                      7
regular_price                   1154
selling_price                   1154
price_valid_from                   0
price_per_unit                  5053
percentage_discount                0
promotion_price                  240
promotion_valid_from               0
promotion_valid_upto               0
promotion_description             28
currency                           1
breadcrumb                       442
pdp_url                        43331
region                             0
pack_size                          0
site_shown_uom                 16283
promotion_description_clean       28
dtype: int64

In [32]:
# private_label_brands = [
#     "Green Choice",
#     "Coles",
#     # "Woofin Good",
#     "KOi",
#     "Im",
#     "Natures Kitchen",
#     "Wellness Road"
# ]

# issues = df[
#     df['brand'].isin(private_label_brands) &
#     (df['brand_type'] != 'Private Label')
# ]

# issues[['brand', 'brand_type', 'product_name', 'pdp_url']]

In [33]:
df.dtypes

unique_id                              object
competitor_name                        object
extraction_date                        object
product_name                           object
grammage_quantity                      object
grammage_unit                          object
regular_price                         float64
selling_price                         float64
price_valid_from                      float64
price_per_unit                         object
percentage_discount                   float64
promotion_price                       float64
promotion_valid_from                  float64
promotion_valid_upto                  float64
promotion_description                  object
currency                               object
breadcrumb                             object
pdp_url                                object
region                                float64
pack_size                             float64
site_shown_uom                         object
promotion_description_clean    str

In [34]:
df.promotion_type.value_counts()

AttributeError: 'DataFrame' object has no attribute 'promotion_type'

In [35]:
dup = df[df.duplicated(
    subset=['unique_id', 'store_name', 'store_postcode'],
    keep=False
)].sort_values(
    ['store_name', 'store_postcode', 'unique_id']
)

dup[['unique_id', 'store_name', 'store_postcode', 'pdp_url']].assign(
    row_no=lambda x: x.index + 2
)

KeyError: Index(['store_postcode', 'store_name'], dtype='object')

In [36]:
df.random_weight_flag.value_counts()

AttributeError: 'DataFrame' object has no attribute 'random_weight_flag'

In [30]:
pattern = r'^[^>]+( > [^>]+)*$'

# Invalid breadcrumbs
invalid = df[
    df["breadcrumb"].isna() |
    (df["breadcrumb"].str.strip() == "") |
    (~df["breadcrumb"].str.fullmatch(pattern, na=False))
]

print(f"Total invalid breadcrumbs: {len(invalid)}")

if not invalid.empty:
    print(invalid[["breadcrumb"]])

Total invalid breadcrumbs: 0


In [31]:
import pandas as pd
import re

price_columns = [
    "grammage_quantity",
    "regular_price",
    "selling_price",
    "price_was",
    "promotion_price",
    "percentage_discount",
    "price_per_unit",
    "multi_buy_items_price_total",
    "multibuy_items_pricesingle"
]

for col in price_columns:

    if col not in df.columns:
        continue

    s = df[col].fillna("").astype(str).str.strip()

    # Extract only numeric-looking part
    # This allows values such as:
    # KG. za 3.58 €
    # KG. za 0.9 €
    number = s.str.extract(r"(\d[\d.,]*)", expand=False).fillna("")

    # --------------------------------------------------
    # 1. MULTIPLE DECIMAL POINTS
    # --------------------------------------------------
    multiple_decimal = number.str.count(r"\.") > 1

    # --------------------------------------------------
    # 2. COMMA USED INSTEAD OF DECIMAL POINT
    # --------------------------------------------------
    comma_decimal = number.str.contains(
        r",",
        regex=True,
        na=False
    )

    # --------------------------------------------------
    # 3. EXACTLY 2 DECIMAL PLACES
    # ONLY regular_price and selling_price
    # --------------------------------------------------
    two_decimal_issue = pd.Series(False, index=df.index)

    if col in ["regular_price", "selling_price"]:
        two_decimal_issue = (
            (s != "") &
            (~s.str.fullmatch(r"\d+\.\d{2}", na=False))
        )

    print(f"\n{'='*15} {col} {'='*15}")

    if multiple_decimal.any():
        print(f"Multiple decimal points: {multiple_decimal.sum()}")
        print(
            df.loc[multiple_decimal, [col]]
            .head(10)
            .to_string(index=False)
        )

    if comma_decimal.any():
        print(f"Comma used instead of dot: {comma_decimal.sum()}")
        print(
            df.loc[comma_decimal, [col]]
            .head(10)
            .to_string(index=False)
        )

    if two_decimal_issue.any():
        print(f"Not in 2-decimal format: {two_decimal_issue.sum()}")
        print(
            df.loc[two_decimal_issue, [col]]
            .head(10)
            .to_string(index=False)
        )


=============== grammage_quantity ===============

=============== regular_price ===============
Not in 2-decimal format: 4835
 regular_price
           5.7
           5.7
           1.5
           5.7
           5.7
          10.0
          39.9
          79.9
           1.6
           5.7

=============== selling_price ===============
Not in 2-decimal format: 4835
 selling_price
           5.7
           5.7
           1.5
           5.7
           5.7
          10.0
          39.9
          79.9
           1.6
           5.7

=============== promotion_price ===============

=============== percentage_discount ===============

=============== price_per_unit ===============


In [32]:
import pandas as pd

price_columns = [
    "grammage_quantity",
    "regular_price",
    "selling_price",
    "price_was",
    "promotion_price",
    "percentage_discount",
    "price_per_unit",
    "multi_buy_items_price_total",
    "multibuy_items_pricesingle"
]

for col in price_columns:
    if col not in df.columns:
        continue

    s = df[col].fillna("").astype(str)

    issues = {
        "Zero values": pd.to_numeric(s, errors="coerce") == 0,
        "Negative values": pd.to_numeric(s, errors="coerce") < 0,
        "Multiple decimal points": s.str.count(r"\.") > 1,
        "More than 2 decimal places": s.str.contains(r"\.\d{3,}$", regex=True),
        "Non-numeric values": (~s.str.match(r"^-?\d+(\.\d+)?$", na=False)) & (s != ""),
        "Leading/Trailing whitespace": s != s.str.strip(),
    }

    print(f"\n{'='*15} {col} {'='*15}")

    for issue, mask in issues.items():
        if mask.any():
            print(f"{issue}: {mask.sum()}")
            print(df.loc[mask, [col]].head(5).to_string(index=False))


=============== grammage_quantity ===============
More than 2 decimal places: 73
grammage_quantity
            0.070
            1.200
            4.700
            0.106
            4.700
Non-numeric values: 189
grammage_quantity
          30 x 10
           41 x 6
          24 x 10
           75 x 3
         5 x 0.45

=============== regular_price ===============

=============== selling_price ===============

=============== promotion_price ===============

=============== percentage_discount ===============

=============== price_per_unit ===============
Multiple decimal points: 1584
 price_per_unit
Per 1 Stk. 0.19
Per 1 Stk. 0.19
Per 1 Stk. 0.33
Per 1 Stk. 0.19
Per 1 Stk. 0.19
Non-numeric values: 25510
price_per_unit
 Per 1 L 13.32
  Per 1 L 6.03
  Per 1 L 5.12
  Per 1 L 3.91
 Per 1 L 11.99


In [50]:
df[df.percentage_discount == 0.0].head(3)

,unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,site_shown_uom,promotion_description_clean


In [33]:
mask = (
    df["price_valid_from"].notna() &
    df["promotion_valid_from"].notna() &
    (df["price_valid_from"] != df["promotion_valid_from"])
)

issues = df.loc[
    mask,
    ["pdp_url", "price_valid_from", "promotion_valid_from"]
].copy()

issues["row_no"] = issues.index + 2

issues[["row_no", "pdp_url", "price_valid_from", "promotion_valid_from"]]

,row_no,pdp_url,price_valid_from,promotion_valid_from


In [34]:
import re

col = "regular_price"

# More than one decimal point in the numeric part
invalid = df[
    df[col]
    .fillna("")
    .astype(str)
    .str.contains(r'\d+\.\d+\.', regex=True)
]

print(f"Rows with multiple decimal points: {len(invalid)}")
print(invalid[[col]])

Rows with multiple decimal points: 0
Empty DataFrame
Columns: [regular_price]
Index: []


In [35]:
import re

url_pattern = re.compile(
    r"^https?://[^\s/$.?#].[^\s]*$",
    re.IGNORECASE
)

invalid_urls = df[
    
    df["pdp_url"].isna() |
    ~df["pdp_url"].astype(str).str.match(url_pattern)
]

print(f"Invalid URLs: {len(invalid_urls)}")
print(invalid_urls[[ "pdp_url"]])

Invalid URLs: 0
Empty DataFrame
Columns: [pdp_url]
Index: []


In [43]:
df.loc[df["pdp_url"].str.contains(r"\s", regex=True, na=False),
       ["pdp_url"]]

,pdp_url


In [36]:
import pandas as pd

# Promotion description has a value containing %
promo_mask = (
    df["promotion_description"].fillna("").astype(str).str.contains("%", na=False)
)

# Percentage discount is empty
discount_empty = (
    df["percentage_discount"].isna() |
    df["percentage_discount"].astype(str).str.strip().eq("")
)

# Both conditions
mask = promo_mask & discount_empty


issues = df.loc[
    mask,
    ["promotion_description", "percentage_discount", "pdp_url"]
].copy()

# Row number (assuming header is row 1)
issues["row_no"] = issues.index + 2

# Arrange columns
issues = issues[
    ["row_no", "pdp_url", "promotion_description", "percentage_discount"]
]

print(issues.to_string(index=False))

Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, percentage_discount]
Index: []


In [37]:
df.grammage_quantity.value_counts()

grammage_quantity
1           15580
0.75         3175
500          1926
100          1855
250          1599
200          1353
150           953
0.5           931
400           853
0.7           799
20            715
300           654
50            588
330           584
350           399
125           388
75            362
1.5           325
180           303
80            294
10            278
60            244
40            238
70            224
450           222
3             219
30            210
2             206
120           197
600           178
750           176
90            173
25            151
240           150
4             150
220           150
375           149
160           141
175           130
140           128
6             125
190           124
800           116
360           113
110           111
5             107
130           103
16             96
8              96
340            96
35             93
225            91
45             85
12             82
18        

In [38]:
mask = (
    df["promotion_valid_from"].notna()
    & df["promotion_valid_from"].astype(str).str.strip().ne("")
    & (
        df["price_valid_from"].isna()
        | df["price_valid_from"].astype(str).str.strip().eq("")
    )
)

issues = df.loc[
    mask,
    
    [ "pdp_url", "promotion_valid_from", "price_valid_from"]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    ["row_no",  "pdp_url", "promotion_valid_from", "price_valid_from"]
]

print(issues.to_string(index=False))

Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_from, price_valid_from]
Index: []


In [39]:
invalid_urls = invalid_urls["pdp_url"].unique()

print("Unique invalid URLs:", len(invalid_urls))
for url in invalid_urls:
    print(repr(url))

Unique invalid URLs: 0


In [40]:
import numpy as np

invalid_urls = np.unique(invalid_urls)

print("Unique invalid URLs:", len(invalid_urls))
for url in invalid_urls:
    print(url)  # Or your specific loop logic


Unique invalid URLs: 0


In [41]:
issues = df[
    df["product_name"].astype(str).str.startswith("Denner", na=False)
    & (df["brand"].fillna("").str.strip() != "Denner")
]

issues[["product_name", "brand", "pdp_url"]]

KeyError: 'brand'

In [42]:
df.instock.value_counts()

AttributeError: 'DataFrame' object has no attribute 'instock'

In [43]:
import re

image_cols = [c for c in df.columns if c.startswith("image_url_")]

# Basic image URL pattern
pattern = re.compile(
    r"^https?://[^\s]+\.(jpg|jpeg|png|webp|gif|bmp|avif)(\?.*)?$",
    re.IGNORECASE
)

for col in image_cols:
    invalid = df[
        
        df[col].notna() &
        (df[col].astype(str).str.strip() != "") &
        ~df[col].astype(str).str.match(pattern)
    ]

    print(f"\n{col}: {len(invalid)} invalid URLs")

    if not invalid.empty:
        print(invalid[["unique_id", col]].head())

In [44]:
(
    df.groupby(['store_addressline2', 'unique_id'])
      .size()
      .reset_index(name='count')
      .query('count > 1')
      .sort_values(['store_addressline2', 'count'], ascending=[True, False])
)

KeyError: 'store_addressline2'

In [45]:
import re

pattern = re.compile(r"^https?://[^\s]+$", re.IGNORECASE)

for col in image_cols:
    invalid = df[
        df[col].notna() &
        (df[col].astype(str).str.strip() != "") &
        ~df[col].astype(str).str.match(pattern)
    ]

    print(f"{col}: {len(invalid)} invalid URLs")

In [52]:
price_cols = [
    "regular_price",
    "selling_price",
   
   
   
]

# Keep only columns that exist in the file
price_cols = [c for c in price_cols if c in df.columns]

# Rows where at least one price column is empty
empty_price_rows = df[df[price_cols].replace("", pd.NA).isna().any(axis=1)]

print(f"Rows with empty price fields: {len(empty_price_rows)}")
display(empty_price_rows[["unique_id","pdp_url", "promotion_description","product_name"] + price_cols])

Rows with empty price fields: 18


,unique_id,pdp_url,promotion_description,product_name,regular_price,selling_price
2021,00-392725,https://shop.billa.at/produkte/goesser-naturradler-alkoholfrei-6x05-00392725,AKTION,"Gösser Naturradler Alkoholfrei 6x0,5 500 ml",NaN,0.89
2296,00-405598,https://shop.billa.at/produkte/edelweiss-hefetrueb-6x05l-00405598,AKTION,"Edelweiss Hefetrüb 6x0,5l 500 ml",NaN,0.72
4096,00-502082,https://shop.billa.at/produkte/goesser-naturradler-6x05l-00502082,AKTION,"Gösser Naturradler 6x0,5l 500 ml",NaN,0.89
4141,00-504120,https://shop.billa.at/produkte/stiegl-paracelsus-bio-zwickl-6x05l-00504120,AKTION,"Stiegl Paracelsus Bio Zwickl 6x0,5l 500 ml",NaN,0.99
4175,00-506941,https://shop.billa.at/produkte/fritzkola-anjola-ananaslimetten-00506941,AKTION,fritz-kola Anjola Ananas-Limetten 1 Liter,NaN,3.79
4177,00-506979,https://shop.billa.at/produkte/fritzkola-limo-honigmelone-00506979,AKTION,fritz-kola Limo Honigmelone 1 Liter,NaN,3.30
5724,00-685017,https://shop.billa.at/produkte/stiegl-weisse-6x05l-00685017,AKTION,"Stiegl Weisse 6x0,5l 500 ml",NaN,0.99
6115,00-707284,https://shop.billa.at/produkte/edelweiss-alkoholfrei-6x05l-00707284,AKTION,"Edelweiss Alkoholfrei 6x0,5l 500 ml",NaN,0.72
6225,00-716803,https://shop.billa.at/produkte/billa-eistee-erdbeermelone-00716803,AKTION,BILLA Eistee Erdbeer-Melone 20 Teebeutel Schachtel,NaN,0.50
7039,00-736178,https://shop.billa.at/produkte/wild-bill-bio-ketchup-mild-00736178,AKTION,Wild Bill Bio Ketchup Mild 280 g Glas,NaN,0.99


In [53]:
import pandas as pd

# Treat empty strings as missing values
promo_price = df["promotion_price"].replace("", pd.NA)
promo_desc = df["promotion_description"].replace("", pd.NA)

# 1. Promotion price present but promotion description missing
price_no_desc = df[
    promo_price.notna() & promo_desc.isna()
]

# 2. Promotion description present but promotion price missing
desc_no_price = df[
    promo_desc.notna() & promo_price.isna()
]

print("Promotion price present but description missing:", len(price_no_desc))
display(price_no_desc[["unique_id", "product_name", "promotion_price","pdp_url", "promotion_description"]])

print("\nPromotion description present but price missing:", len(desc_no_price))
display(desc_no_price[["unique_id", "product_name", "promotion_price", "pdp_url","promotion_description"]])

Promotion price present but description missing: 0


,unique_id,product_name,promotion_price,pdp_url,promotion_description



Promotion description present but price missing: 0


,unique_id,product_name,promotion_price,pdp_url,promotion_description


In [109]:
df.breadcrumb.value_counts()

breadcrumb
Startseite > Make-up > Nägel > Nagellack                                                                    442
Startseite > Make-up > Lippen Make-up > Lippenstift                                                         419
Startseite > Pflege & Parfum > Parfum > Damen Parfum                                                        365
Startseite > Make-up > Teint > Foundation                                                                   295
Startseite > Make-up > Augen Make-up > Augenbrauen                                                          280
Startseite > Haare > Haarpflege > Shampoo                                                                   278
Startseite > Make-up > Augen Make-up > Lidschatten & Paletten                                               251
Startseite > Männerpflege > Körper > Deo                                                                    250
Startseite > Make-up > Augen Make-up > Mascara                                               

In [110]:
invalid = df[df["breadcrumb"].str.contains("…", na=False)]

print(f"Rows with truncated breadcrumbs: {len(invalid)}")
display(invalid[["unique_id", "breadcrumb"]])

Rows with truncated breadcrumbs: 0


,unique_id,breadcrumb


In [111]:
df.competitor_name.value_counts()

competitor_name
dm    17311
Name: count, dtype: int64

In [112]:
df.variants.value_counts()

AttributeError: 'DataFrame' object has no attribute 'variants'

In [54]:
result = df[
    df["selling_price"].isna() & df["regular_price"].isna()

text_cols = df.select_dtypes(include="object").columns

issues = {}

for col in text_cols:
    # Match | not preceded by a backslash
    mask = (
        df[col]
        .fillna("")
        .astype(str)
        .str.contains(r"(?<!\\)\|", regex=True)
    )

    if mask.any():
        issues[col] = df.loc[mask, ["unique_id", col]]

# Print results
if issues:
    print("Columns containing unescaped '|':\n")
    for col, data in issues.items():
        print(f"\n=== {col} ({len(data)} rows) ===")
        print(data.to_string(index=False))
else:
    print("No unescaped '|' found.")
]

print(result)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3466212421.py, line 2)

In [55]:
import pandas as pd

# Text columns only
text_cols = df.select_dtypes(include="object").columns

invalid_chars = [ "\n", "\r", "\t"]

issues = {}

for col in text_cols:
    mask = df[col].fillna("").astype(str).str.contains(r"[\n\r\t]", regex=True)
    if mask.any():
        issues[col] = df.loc[mask, ["unique_id", col]]

# Summary
if issues:
    print("Columns containing \\n, \\r or \\t:\n")
    for col, data in issues.items():
        print(f"{col}: {len(data)} rows")
else:
    print("No \\n, \\r or \\t found.")
    

No \n, \r or \t found.


In [56]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

results = []

for col in text_cols:
    mask = df[col].fillna("").astype(str).str.contains(r"[\n\r\t]", regex=True)

    for idx, value in df.loc[mask, col].items():
        results.append({
            "row_no": idx + 2,      # Excel row number
            "unique_id": df.at[idx, "unique_id"],
            "pdp_url": df.at[idx, "pdp_url"],
            "column": col,
            "issue_value": value
        })

issues_df = pd.DataFrame(results)

issues_df

""


In [69]:
columns = ["product_description", "special_information", "manufacturer_address"]

for col in columns:
    rows = df[
        df[col]
        .fillna("")
        .astype(str)
        .str.contains("|", regex=False, na=False)
    ]

    print(f"\n=== {col}: {len(rows)} rows ===")

    if not rows.empty:
        display(rows[["unique_id", "pdp_url", col]])

KeyError: 'product_description'

In [57]:
import pandas as pd
import re




df["dimensions"] = df["dimensions"].fillna("").str.strip()

pattern = re.compile(
    r'^\d+(\.\d+)?(\s*[a-zA-Z]+)?\s*[xX×]\s*'
    r'\d+(\.\d+)?(\s*[a-zA-Z]+)?\s*[xX×]\s*'
    r'\d+(\.\d+)?(\s*[a-zA-Z]+)?$'
)

invalid = df[
    (df["dimensions"] != "") &
    (~df["dimensions"].str.match(pattern))
]

print(f"Invalid values: {len(invalid)}")
print(invalid[["unique_id", "product_name", "dimensions"]])

KeyError: 'dimensions'

In [58]:
import pandas as pd

issues = []

for col in df.columns:
    if df[col].dtype == "object":
        mask = df[col].fillna("").astype(str).str.contains(
            r'^\s+|\s+$|\t|\n|\r| {2,}',
            regex=True,
            na=False
        )

        if mask.any():
            temp = df.loc[mask, [col]].copy()
            temp["row_no"] = temp.index + 2
            temp["column"] = col
            issues.append(temp[["row_no", "column", col]])

if issues:
    result = pd.concat(issues, ignore_index=True)
    print(result)
else:
    print("No unwanted whitespace found.")

No unwanted whitespace found.


In [100]:

# Replace empty strings with NA
df = df.replace(r'^\s*$', pd.NA, regex=True)

# Filter Fleisch & Fisch
ff = df[df["producthierarchy_level1"] == "Fleisch & Fisch"]

# Subcategories that should have price_per_unit
ppu_subcats = ["Fleisch & Geflügel", "Frischfisch & Schalentiere"]

# Missing price_per_unit
invalid_ppu = ff[
    ff["producthierarchy_level2"].isin(ppu_subcats) &
    ff["price_per_unit"].isna()
]

print("Missing price_per_unit:")
print(invalid_ppu[["unique_id","product_name","producthierarchy_level2","price_per_unit"]])

KeyError: 'producthierarchy_level1'

In [101]:
date_columns = [
    "promotion_valid_upto",
    "promotion_valid_from",
    "price_valid_from"
]

for col in date_columns:
    print("\nColumn:", col)
    print(df[col].value_counts(dropna=False))


Column: promotion_valid_upto
promotion_valid_upto
NaT           3359
2026-11-04     174
2026-11-11     162
2026-12-16      36
2026-12-09      30
2026-11-18      29
2026-10-31      27
2026-12-02      23
2026-11-14      10
2026-10-30       5
2026-11-07       3
2026-12-19       3
2026-12-05       2
Name: count, dtype: int64

Column: promotion_valid_from
promotion_valid_from
NaT           3119
2026-10-29     359
2026-11-09      94
2026-11-05      83
2026-11-02      73
2026-12-03      31
2026-11-16      29
2026-11-30      25
2026-12-14      15
2026-11-19      11
2026-10-22       6
2026-10-26       5
2026-12-10       4
2026-11-26       4
2026-11-12       4
2026-12-17       1
Name: count, dtype: int64

Column: price_valid_from
price_valid_from
NaN           3119
29.10.2026     359
9.11.2026       94
5.11.2026       83
2.11.2026       73
3.12.2026       31
16.11.2026      29
30.11.2026      25
14.12.2026      15
19.11.2026      11
22.10.2026       6
26.10.2026       5
10.12.2026       4
26.11

In [60]:
invalid_discount = df[
    df["percentage_discount"]
    .astype(str)
    .str.contains(r"[%\-]", regex=True, na=False)
]

print("Invalid percentage_discount values:")
print(invalid_discount[["unique_id", "product_name", "percentage_discount"]])

print("\nCount:", len(invalid_discount))

Invalid percentage_discount values:
Empty DataFrame
Columns: [unique_id, product_name, percentage_discount]
Index: []

Count: 0


In [61]:
s = df['percentage_discount']

mask = (
    s.notna() &
    s.astype(str).str.strip().ne("") &
    ~s.astype(str).str.fullmatch(r'\d+', na=False)
)

issues = df.loc[
    mask,
    ['pdp_url', 'percentage_discount']
].copy()

issues['row_no'] = issues.index + 2

issues = issues[['row_no', 'pdp_url', 'percentage_discount']]

print(issues.to_string(index=False))
print(f"\nTotal issues: {len(issues)}")

Empty DataFrame
Columns: [row_no, pdp_url, percentage_discount]
Index: []

Total issues: 0


In [102]:
invalid = df[
    df["promotion_description"].notna() &
    (
        df["promotion_valid_from"].isna() |
        df["promotion_valid_upto"].isna()
    )
]

print(f"Rows with promotion description but missing promotion dates: {len(invalid)}")

display(
    invalid[
        [
            "promotion_price",
            "promotion_description",
            "promotion_valid_from",
            "promotion_valid_upto",
            "pdp_url"
        ]
    ]
)

Rows with promotion description but missing promotion dates: 248


,promotion_price,promotion_description,promotion_valid_from,promotion_valid_upto,pdp_url
25,9.99,-14%,2026-10-29,NaT,https://sortiment.lidl.ch/de/catalog/product/view/id/15653/s/trutenbrust-filet-0184353/category/54/
51,NaN,Aktion,NaT,NaT,https://sortiment.lidl.ch/de/catalog/product/view/id/22433/s/kleinkerze-mit-holzdocht-1024397/category/54/
54,NaN,Aktion,NaT,NaT,https://sortiment.lidl.ch/de/catalog/product/view/id/22432/s/ferrero-kinder-ueberraschung-0128194/category/54/
55,NaN,Aktion,NaT,NaT,https://sortiment.lidl.ch/de/catalog/product/view/id/22431/s/ferrero-kinder-ueberraschung-0128196/category/54/
56,NaN,Aktion,NaT,NaT,https://sortiment.lidl.ch/de/catalog/product/view/id/22430/s/ferrero-kinder-schokolade-weihnachtsmann-0026820/category/54/
57,NaN,Aktion,NaT,NaT,https://sortiment.lidl.ch/de/catalog/product/view/id/22429/s/duplo-knusperlen-1012097/category/54/
58,NaN,Aktion,NaT,NaT,https://sortiment.lidl.ch/de/catalog/product/view/id/22428/s/nutella-winterstiefel-1007201/category/54/
59,NaN,Aktion,NaT,NaT,https://sortiment.lidl.ch/de/catalog/product/view/id/22427/s/grand-ferrero-rocher-0117291/category/54/
60,NaN,Aktion,NaT,NaT,https://sortiment.lidl.ch/de/catalog/product/view/id/22426/s/geriebener-trueffel-5700777/category/54/
61,NaN,Aktion,NaT,NaT,https://sortiment.lidl.ch/de/catalog/product/view/id/22424/s/italienisches-risotto-0022571/category/54/


In [124]:
mask = (
    df["regular_price"].notna() &
    df["price_was"].notna() &
    (pd.to_numeric(df["regular_price"], errors="coerce") !=
     pd.to_numeric(df["price_was"], errors="coerce"))
)

issues = df.loc[mask, ["pdp_url", "regular_price", "price_was"]].copy()
issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "regular_price", "price_was"]]

issues

KeyError: 'price_was'

In [62]:
mask = (
    df["unique_id"].astype(str).fillna("") + "P"
    !=
    df["product_unique_key"].astype(str).fillna("")
)

issues = df.loc[mask, ["unique_id", "product_unique_key", "pdp_url"]].copy()
issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "unique_id", "product_unique_key"]]

issues

KeyError: 'product_unique_key'

In [63]:
import pandas as pd

# Normalize for case-insensitive comparison
df["site_shown_uom"] = df["site_shown_uom"].fillna("").astype(str)
df["grammage_unit"] = df["grammage_unit"].fillna("").astype(str)
df["grammage_quantity"] = df["grammage_quantity"].fillna("").astype(str)

# -----------------------------
# Rule 1: Teebeutel / Beutel -> btl
# -----------------------------
btl_keywords = [
    "Teebeutel Packung",
    "Teebeutel",
    "Beutel",
    "Btl",
    "Teebeutel Karton",
    "Teebeutel Paket",
    "Packung Beutel"
]

btl_invalid = df[
    df["site_shown_uom"].str.contains("|".join(btl_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "btl")
]

print(f"BTL unit mismatches: {len(btl_invalid)}")
display(btl_invalid[[
    "pdp_url",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit"
]])

# -----------------------------
# Rule 2: Portion Packung / ANW -> stuck
# -----------------------------
stuck_keywords = [
    "Portion Packung",
    "ANW"
]

stuck_invalid = df[
    df["site_shown_uom"].str.contains("|".join(stuck_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "stuck")
]

print(f"STUCK unit mismatches: {len(stuck_invalid)}")
display(stuck_invalid[[
    "unique_id",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit"
]])

# -----------------------------
# Rule 3: wg / Waschgänge -> wg
# -----------------------------
wg_keywords = [
    "wg",
    "Waschgänge"
]

wg_invalid = df[
    df["site_shown_uom"].str.contains("|".join(wg_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "wg")
]

print(f"WG unit mismatches: {len(wg_invalid)}")
display(wg_invalid[[
    "unique_id",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit",
    "pdp_url"
]])

BTL unit mismatches: 987


,pdp_url,product_name,site_shown_uom,grammage_quantity,grammage_unit
8,https://shop.billa.at/produkte/toppits-gefrierbeutel-6-liter-00101513,Toppits Gefrierbeutel 6 Liter 20 Beutel Packung,20 Beutel Packung,1,stück
13,https://shop.billa.at/produkte/toppits-gefrierbeutel-1l-00101511,Toppits Gefrierbeutel 1L 40 Beutel Packung,40 Beutel Packung,1,stück
37,https://shop.billa.at/produkte/vegeta-wuerzmischung-00103992,Vegeta Würzmischung 1 kg Beutel,1 kg Beutel,1,kg
58,https://shop.billa.at/produkte/knorr-basis-fuer-faschiertes-0011080,Knorr Basis für Faschiertes 76 g Beutel,76 g Beutel,76,g
60,https://shop.billa.at/produkte/ricola-schweizer-kraeuterbonbon-001404,Ricola Schweizer Kräuterbonbon 75 g Beutel,75 g Beutel,75,g
61,https://shop.billa.at/produkte/ricola-salbei-zuckerfrei-00141657,Ricola Salbei zuckerfrei 75 g Beutel,75 g Beutel,75,g
74,https://shop.billa.at/produkte/gloria-watte-0014270,Gloria Watte 200 g Beutel,200 g Beutel,200,g
88,https://shop.billa.at/produkte/ricola-zitronenmelisse-zuckerfrei-00144516,Ricola Zitronenmelisse zuckerfrei 75 g Beutel,75 g Beutel,75,g
166,https://shop.billa.at/produkte/kinder-schokobons-00116129,Kinder Schokobons 200 g Beutel,200 g Beutel,200,g
198,https://shop.billa.at/produkte/nimm2-vitaminbonbons-0017772,nimm2 Vitaminbonbons 240 g Beutel,240 g Beutel,240,g


STUCK unit mismatches: 0


,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit


WG unit mismatches: 0


,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit,pdp_url


In [64]:
import pandas as pd

price_cols = ["regular_price", "selling_price"]

for col in price_cols:
    # Ignore missing values
    mask = (
        df[col].notna() &
        (~df[col].astype(float).round(2).eq(df[col].astype(float)))
    )

    invalid = df.loc[mask, ["unique_id", "product_name", col]]

    print(f"\n{col}: {len(invalid)} values not in 2-decimal format")
    display(invalid)


regular_price: 0 values not in 2-decimal format


,unique_id,product_name,regular_price



selling_price: 0 values not in 2-decimal format


,unique_id,product_name,selling_price


In [65]:
import pandas as pd

# Match both old and new site keywords
promo_mask = (
    df["promotion_description"]
    .fillna("")
    .astype(str)
    .str.contains(r"Trajno znižano|Znižano", case=False, na=False)
)

# Empty promotion price
promo_price_empty = (
    df["promotion_price"].isna()
    | df["promotion_price"].astype(str).str.strip().eq("")
)

# Check where promotion price is NOT empty
promo_price_issue = promo_mask & ~promo_price_empty

# Get all matching records
issues = df.loc[
    promo_mask,
    [
        "unique_id",
        "pdp_url",
        "promotion_description",
        "regular_price",
        "selling_price",
        "promotion_price"
    ]
].copy()

issues["row_no"] = issues.index + 2

# Add issue remarks
def check_issue(row):
    remarks = []

    if pd.notna(row["promotion_price"]) and str(row["promotion_price"]).strip() != "":
        remarks.append("promotion_price should be empty")

    if row["regular_price"] != row["selling_price"]:
        remarks.append("regular_price and selling_price are not equal")

    return "; ".join(remarks)

issues["issue"] = issues.apply(check_issue, axis=1)

# Show only violations
issues = issues[issues["issue"] != ""]

issues = issues[
    [
        "row_no",
        "unique_id",
        "pdp_url",
        "promotion_description",
        "regular_price",
        "selling_price",
        "promotion_price",
        "issue"
    ]
]

print(issues.to_string(index=False))

Empty DataFrame
Columns: [row_no, unique_id, pdp_url, promotion_description, regular_price, selling_price, promotion_price, issue]
Index: []


In [66]:
import re
import pandas as pd

# Normalize grammage_unit
df["grammage_unit_norm"] = df["grammage_unit"].fillna("").str.lower().str.strip()

# Extract quantity from site_shown_uom
df["site_qty"] = (
    df["site_shown_uom"]
      .str.extract(r"(\d+(?:[.,]\d+)?)", expand=False)
      .str.replace(",", ".", regex=False)
)

df["site_qty"] = pd.to_numeric(df["site_qty"], errors="coerce")

# ---------- Rule 1 : Beutel / Btl ----------
btl_pattern = (
    r"Teebeutel Packung|Teebeutel Karton|Teebeutel Paket|"
    r"Packung Beutel|Teebeutel|Beutel|Btl"
)

btl_issue = df[
    df["site_shown_uom"].fillna("").str.contains(btl_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "btl") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

# ---------- Rule 2 : Portion Packung / ANW ----------
stuck_pattern = r"Portion Packung|ANW"

stuck_issue = df[
    df["site_shown_uom"].fillna("").str.contains(stuck_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "stuck") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

# ---------- Rule 3 : Waschgang / Waschgänge / wg ----------
wg_pattern = r"\bwg\b|Waschgänge|Waschgang"

wg_issue = df[
    df["site_shown_uom"].fillna("").str.contains(wg_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "wg") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

print("Btl issues:", len(btl_issue))
print("Stuck issues:", len(stuck_issue))
print("WG issues:", len(wg_issue))

Btl issues: 1141
Stuck issues: 0
WG issues: 76


In [130]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

results = []

for col in text_cols:
    mask = df[col].fillna("").astype(str).str.contains(r" {2,}", regex=True)

    for idx, value in df.loc[mask, col].items():
        results.append({
            "row_no": idx + 2,
            "pdp_url": df.at[idx, "pdp_url"],
            "column": col,
            "issue_value": repr(value)
        })

issues_df = pd.DataFrame(results)
issues_df

,row_no,pdp_url,column,issue_value
0,476,https://www.dm.at/p/d/3146833/dmbio-kraeutertee-deine-innere-ruhe-mit-kamille-minze-lavendel-und-ashwagandha-20-x-1-8g,product_name,"'Kräutertee, Deine innere Ruhe mit Kamille, Minze, Lavendel & Ashwagandha (20 x 1,8g), 36 g'"
1,1650,https://www.dm.at/p/d/1712323/reell-e-toenungsshampoo-7-4-kupfergold,product_name,"'Tönungsshampoo 7.4 Kupfergold, 14 ml'"
2,2061,https://www.dm.at/p/d/1556953/syoss-augenbrauenfarbe-dunkelbraun-4-1,product_name,"'Augenbrauenfarbe Dunkelbraun 4-1, 1 St'"
3,2359,https://www.dm.at/p/d/3104287/mivolis-tabletten-vitamin-b12-350,product_name,"'Tabletten Vitamin B12 350, 30 St'"
4,2465,https://www.dm.at/p/d/1647682/manhattan-cosmetics-khol-kajal-eyemazing-51d-nude-couture,product_name,"'Khol Kajal Eyemazing 51D Nude Couture, 1 St'"
5,2870,https://www.dm.at/p/d/1536732/real-techniques-make-up-schwamm-miracle-complexion-mit-case,product_name,"'Make-Up Schwamm Miracle Complexion mit Case, 1 St'"
6,3289,https://www.dm.at/p/d/3155846/essence-lippenstift-gloss-stick-caramelted-01-sweet-lips-caramel-kiss,product_name,"'Lippenstift Gloss Stick Caramelted 01 Sweet Lips, Caramel Kiss, 3 g'"
7,3536,https://www.dm.at/p/d/1612662/artdeco-lippenstift-hydra-care-02-charming-oasis,product_name,"'Lippenstift Hydra Care 02 Charming Oasis, 3,5 g'"
8,3648,https://www.dm.at/p/d/1657019/maybelline-new-york-foundation-4in1-instant-perfector-glow-00-fair-light,product_name,"'Foundation 4in1 Instant Perfector Glow 00 Fair Light, 20 ml'"
9,3760,https://www.dm.at/p/d/1300800/maybelline-new-york-lippenstift-super-stay-24h-615-soft-taupe,product_name,"'Lippenstift Super Stay 24H 615 Soft Taupe, 5 g'"


In [131]:
btl_issue[
    [
        "unique_id",
        "product_name",
        "site_shown_uom",
        "grammage_quantity",
        "grammage_unit",
        "site_qty"
    ]
].head(20)

,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit,site_qty
3,4008137030941,"Manuka Salbei Tee (20 x 2g), 40 g",20 Btl,20,btl,20.0
9,4008137030903,"Kräutertee Stoffwechsel Talent mit Zitrone, Apfelessig und Ingwer (20 x 1,8g), 36 g",20 Btl,20,btl,20.0
13,4008137030484,"Kräutertee, Menstruations Tee (12 x 2g), 24 g",12 Btl,12,btl,12.0
474,4070765083774,"Kräutertee, Deine innere Ruhe mit Kamille, Minze, Lavendel & Ashwagandha (20 x 1,8g), 36 g",20 Btl,20,btl,20.0
481,4070765087109,"Gewürztee, Chai Vanille mit Zimt, Ingwer & Vanille (20 x 2g), 40 g",20 Btl,20,btl,20.0
482,4070765122374,"Grüner Tee Matcha Vanille Sticks (6 x 9g), 54 g",6 Btl,6,btl,6.0
484,4070765087123,"Kräutertee Deine Leichtigkeit mit Brennnessel, Grüner Tee & Fenchel (20 x 2g), 40 g",20 Btl,20,btl,20.0
486,4070765105360,"Wintertee Früchtemischung (20 x 2g), 40 g",20 Btl,20,btl,20.0
935,4070765058802,"Grüner Tee mit 5% Japanischer Matcha (20 × 1,5 g), 30 g",20 Btl,20,btl,20.0
1466,4066447922905,"Früchtetee Hagebutte Hibiskus, 40 g",20 Btl,20,btl,20.0


In [132]:
wrong_unit_wg = wg_issue[
    wg_issue["grammage_unit_norm"] != "wg"
]

wrong_qty_wg = wg_issue[
    (wg_issue["grammage_unit_norm"] == "wg") &
    (wg_issue["grammage_quantity"] != wg_issue["site_qty"])
]

print("WG - Wrong unit:", len(wrong_unit_wg))
print("WG - Wrong quantity:", len(wrong_qty_wg))

WG - Wrong unit: 0
WG - Wrong quantity: 1


In [133]:
wrong_unit_wg[
    [
        "unique_id",
        "product_name",
        "site_shown_uom",
        "grammage_quantity",
        "grammage_unit",
        "pdp_url"
    ]
]

,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit,pdp_url


In [134]:
# JO PREIS price structure
jo_mask = df['promotion_description'].fillna('').astype(str).str.contains(
    r'JO PREIS',
    case=False,
    na=False
)

issues = df.loc[
    jo_mask & (
        df['regular_price'].isna() |
        df['selling_price'].isna() |
        df['price_per_unit'].isna()
    ),
    [
        'pdp_url',
        'promotion_description',
        'regular_price',
        'selling_price',
        'price_per_unit'
    ]
].copy()

issues['row_no'] = issues.index + 2

issues = issues[
    ['row_no', 'pdp_url', 'promotion_description',
     'regular_price', 'selling_price', 'price_per_unit']
]

print(f"JO PREIS issues: {len(issues)}")
print(issues.to_string(index=False))

JO PREIS issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, regular_price, selling_price, price_per_unit]
Index: []


In [67]:
text_cols = df.select_dtypes(include="object").columns

leading_trailing = {}

for col in text_cols:
    mask = (
        df[col].fillna("").astype(str)
        != df[col].fillna("").astype(str).str.strip()
    )
    if mask.any():
        leading_trailing[col] = df.loc[mask, col]

print(leading_trailing.keys())

dict_keys([])


In [68]:
multiple_spaces = {}

for col in text_cols:
    mask = df[col].fillna("").str.contains(r"\s{2,}", regex=True)
    if mask.any():
        multiple_spaces[col] = df.loc[mask, col]

print(multiple_spaces.keys())

dict_keys([])


In [69]:
special_space = {}

for col in text_cols:
    mask = df[col].fillna("").str.contains(r"[\t\r\n]", regex=True)
    if mask.any():
        special_space[col] = df.loc[mask, col]

print(special_space.keys())

dict_keys([])


In [70]:
special_space

{}

In [71]:
issues = []

for col in text_cols:
    mask = (
        (df[col].fillna("") != df[col].fillna("").str.strip()) |
        df[col].fillna("").str.contains(r"\s{2,}", regex=True) |
        df[col].fillna("").str.contains(r"[\t\r\n]", regex=True) |
        df[col].fillna("").str.contains("\u00A0", regex=False)
    )

    if mask.any():
        tmp = df.loc[mask, ["unique_id", col]].copy()
        tmp["column"] = col
        issues.append(tmp)

issues = pd.concat(issues, ignore_index=True)

issues.head()

,unique_id,grammage_quantity,column,site_shown_uom
0,00-10763,1 000,grammage_quantity,NaN
1,00-14030,1 000,grammage_quantity,NaN
2,00-15389,1 000,grammage_quantity,NaN
3,00-13418,1 000,grammage_quantity,NaN
4,00-19467,1 400,grammage_quantity,NaN


In [72]:
import pandas as pd

# Convert to datetime for proper date comparison
price_date = pd.to_datetime(df["price_valid_from"], errors="coerce")
promo_date = pd.to_datetime(df["promotion_valid_from"], errors="coerce")

# Find rows where both have values but are different
mask = (
    price_date.notna()
    & promo_date.notna()
    & (price_date != promo_date)
)

issues = df.loc[
    mask,
    ["unique_id", "pdp_url", "price_valid_from", "promotion_valid_from"]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    ["row_no", "unique_id", "pdp_url",
     "price_valid_from", "promotion_valid_from"]
]

print(issues.to_string(index=False))
print(f"\nTotal issues: {len(issues)}")

Empty DataFrame
Columns: [row_no, unique_id, pdp_url, price_valid_from, promotion_valid_from]
Index: []

Total issues: 0


In [73]:
import pandas as pd

# Convert price columns to numeric
cols = ["regular_price", "selling_price", "promotion_price", "price_per_unit"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

invalid = df[
    # Mehrweg product
    (
        df["product_name"].fillna("").str.contains("Mehrweg", case=False) |
        df["breadcrumb"].fillna("").str.contains("Mehrweg", case=False)
    )
    &
    # Has AKTION promotion
    df["promotion_description"].fillna("").str.contains("AKTION", case=False)
    &
    # Rule violation
    (
        df["regular_price"].notna() |
        (df["selling_price"] != df["price_per_unit"]) |
        (df["promotion_price"] != df["price_per_unit"])
    )
]

print("Invalid rows:", len(invalid))

invalid[
    [
        "unique_id",
        "product_name",
        "regular_price",
        "selling_price",
        "promotion_price",
        "price_per_unit",
        "promotion_description",
        "pdp_url",
    ]
]

Invalid rows: 0


,unique_id,product_name,regular_price,selling_price,promotion_price,price_per_unit,promotion_description,pdp_url


In [74]:
import pandas as pd

# 1. Identify which IDs are duplicated
duplicate_ids = df[df.duplicated(subset=['unique_id'], keep=False)]

# 2. Count occurrences of each duplicate ID
duplicate_counts = df['unique_id'].value_counts()
duplicate_counts = duplicate_counts[duplicate_counts > 1]


In [143]:
duplicate_ids

,unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,site_shown_uom,promotion_description_clean,grammage_unit_norm,site_qty


In [75]:
df.grammage_unit.value_counts()

grammage_unit
g        7303
l        2062
ml       1132
stück    1077
kg        396
btl       154
wg          4
Name: count, dtype: int64

In [145]:
df[df.grammage_unit == "stuck"]

,unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,site_shown_uom,promotion_description_clean,grammage_unit_norm,site_qty


In [146]:
duplicates = df[
    df.duplicated(subset=["store_addressline1", "unique_id"], keep=False)
].sort_values(["store_addressline1", "unique_id"])

print("Duplicate unique_id within stores:", len(duplicates))

display(duplicates[["store_addressline1", "unique_id"]])


KeyError: Index(['store_addressline1'], dtype='object')

In [147]:
df.dtypes

unique_id                               int64
competitor_name                        object
extraction_date                        object
product_name                           object
grammage_quantity                      object
grammage_unit                          object
regular_price                         float64
selling_price                         float64
price_valid_from                       object
price_per_unit                        float64
percentage_discount                   float64
promotion_price                       float64
promotion_valid_from                   object
promotion_valid_upto                  float64
promotion_description                 float64
currency                               object
breadcrumb                             object
pdp_url                                object
region                                float64
pack_size                             float64
site_shown_uom                         object
promotion_description_clean    str

In [148]:
issues = df[
    df["grammage_quantity"].astype(str).str.count(r"\d+\s*x") > 1
][["grammage_quantity", "pdp_url"]].reset_index(names="row_no")

issues

,row_no,grammage_quantity,pdp_url


In [96]:
wine = df[
    df["producthierarchy_level1"].astype(str).str.contains("wein|wine", case=False, na=False)
]

issues = wine[
    wine["site_shown_uom"].astype(str).str.contains(r"\b[2-9]\d*\s*x\b", regex=True, na=False)
][["site_shown_uom", "regular_price", "selling_price", "pdp_url"]]

issues.reset_index(names="row_no")

KeyError: 'producthierarchy_level1'

In [76]:
import pandas as pd

issues = []

for idx, row in df.iterrows():

    rp = str(row.get("regular_price", "")).strip()
    sp = str(row.get("selling_price", "")).strip()
    pp = str(row.get("promotion_price", "")).strip()
    pdesc = str(row.get("promotion_description", "")).strip()

    # Ignore NaN represented as string
    if rp.lower() == "nan": rp = ""
    if sp.lower() == "nan": sp = ""
    if pp.lower() == "nan": pp = ""
    if pdesc.lower() == "nan": pdesc = ""

    remarks = []

    # Rule 1: Regular price contains comma
    if "," in rp:
        remarks.append("Regular price contains comma (,) instead of dot (.)")

    # Rule 2: Regular price missing
    if rp == "" and sp != "":
        remarks.append("Regular price missing (should be assigned from selling price)")

    # Rule 3: Promotion description missing but prices exist
    if pdesc == "" and rp != "" and sp != "":
        if pp != sp:
            remarks.append("Promotion description missing - promotion_price should equal selling_price")

    # Rule 4: Aktion products with all prices (reference only)
    if "aktion" in pdesc.lower():
        if rp == "":
            remarks.append("Aktion product with missing regular price (requires previous iteration data)")
        elif pp == "":
            remarks.append("Aktion product missing promotion_price")

    if remarks:
        issues.append({
            "row_no": idx + 1,
            "profile_url": row.get("profile_url", ""),
            "regular_price": rp,
            "selling_price": sp,
            "promotion_price": pp,
            "promotion_description": pdesc,
            "issue": " | ".join(remarks)
        })

issues_df = pd.DataFrame(issues)

print(f"Total issues found: {len(issues_df)}")
display(issues_df)

Total issues found: 10957


,row_no,profile_url,regular_price,selling_price,promotion_price,promotion_description,issue
0,2,,3.39,3.39,,,Promotion description missing - promotion_price should equal selling_price
1,3,,2.49,2.49,,,Promotion description missing - promotion_price should equal selling_price
2,4,,1.49,1.49,,,Promotion description missing - promotion_price should equal selling_price
3,5,,3.19,3.19,,,Promotion description missing - promotion_price should equal selling_price
4,6,,2.95,2.95,,,Promotion description missing - promotion_price should equal selling_price
5,7,,2.99,2.99,,,Promotion description missing - promotion_price should equal selling_price
6,8,,2.99,2.99,,,Promotion description missing - promotion_price should equal selling_price
7,9,,2.99,2.99,,,Promotion description missing - promotion_price should equal selling_price
8,10,,4.29,4.29,,,Promotion description missing - promotion_price should equal selling_price
9,11,,2.59,2.59,,,Promotion description missing - promotion_price should equal selling_price


In [234]:
df[df.selling_price.isna()].pdp_url

Series([], Name: pdp_url, dtype: object)

In [99]:
mask = (
    df['selling_price'].eq(df['regular_price']) &
    df['selling_price'].eq(df['promotion_price'])
)

issues = df.loc[mask, ['pdp_url', 'selling_price', 'regular_price', "promotion_description", 'promotion_price']].copy()
issues['row_no'] = issues.index + 2


issues[['row_no', 'pdp_url', 'selling_price', 'regular_price', "promotion_description", 'promotion_price']]

,row_no,pdp_url,selling_price,regular_price,promotion_description,promotion_price


In [98]:
# Find Mehrweg products that are not yogurt
# and have missing price_per_unit

mehrweg_mask = (
    df['promotion_description']
    .fillna('')
    .astype(str)
    .str.contains('Mehrweg', case=False, na=False)
)

# Exclude yogurt products
not_yogurt_mask = ~df['product_name'].fillna('').astype(str).str.contains(
    'yogurt|joghurt|yoghurt',
    case=False,
    na=False
)

# price_per_unit is missing / empty
price_per_unit_missing = (
    df['price_per_unit'].isna() |
    df['price_per_unit'].astype(str).str.strip().eq('')
)

mask = mehrweg_mask & not_yogurt_mask & price_per_unit_missing

issues = df.loc[
    mask,
    ['pdp_url', 'product_name', 'promotion_description', 'price_per_unit']
].copy()

# Excel row number
issues['row_no'] = issues.index + 2

issues = issues[
    ['row_no', 'pdp_url', 'product_name',
     'promotion_description', 'price_per_unit']
]

print(f"Total Mehrweg issues: {len(issues)}")
print(issues.to_string(index=False))

Total Mehrweg issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, product_name, promotion_description, price_per_unit]
Index: []


In [97]:
# Check if selling_price or promotion_price is greater than regular_price

regular_price = pd.to_numeric(df['regular_price'], errors='coerce')
selling_price = pd.to_numeric(df['selling_price'], errors='coerce')
promotion_price = pd.to_numeric(df['promotion_price'], errors='coerce')

mask = (
    regular_price.notna() &
    (
        (selling_price.notna() & (selling_price > regular_price)) |
        (promotion_price.notna() & (promotion_price > regular_price))
    )
)

issues = df.loc[
    mask,
    ['pdp_url', 'promotion_description',
     'regular_price', 'selling_price', 'promotion_price']
].copy()

# Excel row number
issues['row_no'] = issues.index + 2

issues = issues
    ['row_no', 'pdp_url', 'promotion_description',
     'regular_price', 'selling_price', 'promotion_price']

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

IndentationError: unexpected indent (410566826.py, line 25)

In [107]:
mask = (
    df["regular_price"].notna() &
    df["price_was"].notna() &
    (pd.to_numeric(df["regular_price"], errors="coerce") ==
     pd.to_numeric(df["price_was"], errors="coerce"))
)

issues = df.loc[mask, ["pdp_url", "regular_price", "price_was"]].copy()
issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "regular_price", "price_was"]]

issues

KeyError: 'price_was'

In [108]:
df.organictype.value_counts()

AttributeError: 'DataFrame' object has no attribute 'organictype'

In [109]:
for i, ch in enumerate(url):
    if ch == " ":
        print(i, "Normal space")
    elif ch == "\xa0":
        print(i, "NBSP")

NameError: name 'url' is not defined

In [110]:
issues = df[
    df["product_unique_key"].astype(str) != (df["unique_id"].astype(str) + "P")
][["unique_id", "product_unique_key", "pdp_url"]].reset_index(names="row_no")

issues


KeyError: 'product_unique_key'

In [111]:
invalid_urls = [u.replace(" ", "\xa0") for u in invalid_urls]

issues = df[df["pdp_url"].isin(invalid_urls)][
    ["unique_id", "pdp_url"]
].reset_index(names="row_no")

issues

,row_no,unique_id,pdp_url


In [112]:
df[df['site_shown_uom'] == ""]


,unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,site_shown_uom,promotion_description_clean,grammage_unit_norm,site_qty


In [113]:
import pandas as pd

issues = []

for col in df.select_dtypes(include="object").columns:

    mask = df[col].fillna("").astype(str).str.contains(
        r'^\s+|\s+$|\t|\n|\r| {2,}',
        regex=True,
        na=False
    )

    if mask.any():
        temp = df.loc[mask, [col, "pdp_url"]].copy()

        temp["row_no"] = temp.index + 2
        temp["column"] = col
        temp["issue_value"] = temp[col]

        issues.append(
            temp[["row_no", "pdp_url", "column", "issue_value"]]
        )

if issues:
    result = pd.concat(issues, ignore_index=True)

    print(f"Total whitespace issues: {len(result)}")
    print(result.to_string(index=False))
else:
    print("No unwanted whitespace found.")

Total whitespace issues: 524
 row_no                                                                                                                                        pdp_url       column                                                                                                issue_value
    118                                          https://www.dm.si/p/d/3143521/balea-beauty-expert-vlazilni-gel-za-prhanje-s-25-glicerola-in-1-betaina product_name                             BEAUTY EXPERT vlažilni gel  za prhanje s 25 % glicerola in 1 % betaina, 200 ml
    238                                                                            https://www.dm.si/p/d/1321849/essence-maskara-i-love-extreme-volume product_name                                                                      Maskara  I love extreme volume, 12 ml
   2352                                                                      https://www.dm.si/p/d/2244306/afrodita-young-pure-gel-za-susenje-mozoljev product

In [114]:
df.grammage_quantity.value_counts()

grammage_quantity
1.0       4445
50.0       547
100.0      478
200.0      392
30.0       373
250.0      365
150.0      353
500.0      248
75.0       225
400.0      184
300.0      184
10.0       161
60.0       157
2.0        151
20.0       146
40.0       122
3.0         82
15.0        78
25.0        72
125.0       71
4.0         71
8.0         67
180.0       65
6.0         59
12.0        58
16.0        52
120.0       51
80.0        48
90.0        47
0.5         44
5.0         43
330.0       42
70.0        36
0.75        36
750.0       34
190.0       33
85.0        30
18.0        30
350.0       29
1000.0      29
45.0        25
4.8         23
35.0        23
32.0        20
240.0       20
55.0        20
24.0        19
14.0        19
800.0       17
65.0        16
28.0        16
230.0       15
36.0        14
42.0        13
1.5         13
360.0       13
7.0         12
220.0       12
340.0       12
320.0       11
46.0        11
236.0       10
1200.0      10
160.0       10
135.0       10
9.0    

In [94]:
import pandas as pd
import re

issues = []

# Check all text/object columns
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    s = df[col].fillna("").astype(str)

    # 1. Value ending with comma
    mask_comma = s.str.rstrip().str.endswith(",")

    if mask_comma.any():
        temp = df.loc[mask_comma, [col]].copy()
        temp["row_no"] = temp.index + 2
        temp["column"] = col
        temp["issue"] = "Value ends with comma (,)"
        temp["issue_value"] = temp[col]

        # Add PDP URL if available
        if "pdp_url" in df.columns:
            temp["URL"] = df.loc[mask_comma, "pdp_url"]

        issues.append(
            temp[["row_no", "URL", "column", "issue_value", "issue"]]
        )

    # 2. Leading/trailing whitespace
    mask_space = s.ne(s.str.strip()) & s.ne("")

    if mask_space.any():
        temp = df.loc[mask_space, [col]].copy()
        temp["row_no"] = temp.index + 2
        temp["column"] = col
        temp["issue"] = "Leading/trailing whitespace"
        temp["issue_value"] = temp[col]

        if "pdp_url" in df.columns:
            temp["URL"] = df.loc[mask_space, "pdp_url"]

        issues.append(
            temp[["row_no", "URL", "column", "issue_value", "issue"]]
        )

# Combine results
if issues:
    result = pd.concat(issues, ignore_index=True)

    print(result.to_string(index=False))
else:
    print("No issues found.")

No issues found.


In [116]:
mask = (
    df["product_name"].astype("string").str.startswith("Denner", na=False)
    & df["brand"].astype("string").str.strip().ne("Denner")
)

result = df.loc[mask, ["unique_id", "pdp_url", "product_name", "brand"]].copy()
result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[["row_no", "unique_id", "pdp_url", "product_name", "brand"]].to_string(index=False))

KeyError: 'brand'

In [117]:
mask = (
    df["file_name"].notna()
    & df["file_name"].astype(str).str.strip().ne("")
)

result = df.loc[mask, ["unique_id", "pdp_url", "file_name"]].copy()
result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[["row_no", "unique_id", "pdp_url", "file_name"]].to_string(index=False))

KeyError: 'file_name'

In [93]:
price_cols = ["regular_price"]

for col in price_cols:
    # Check exactly 2 decimal places
    mask = (
        df[col].notna() &
        ~df[col].astype(str).str.match(r'^\d+\.\d{2}$')
    )

    if mask.any():
        issues = df.loc[mask, ['pdp_url', col]].copy()

        # Row number = dataframe index + 2
        issues.insert(0, 'row_no', issues.index + 2)

        print(f"\n=== {col} - Invalid 2 Decimal Format ===")
        print(issues.to_string(index=False))
        print(f"\nTotal issues: {len(issues)}")

    else:
        print(f"{col}: All values are in 2 decimal format.")


=== regular_price - Invalid 2 Decimal Format ===
 row_no                                                                                                                 pdp_url  regular_price
    130         https://sortiment.lidl.ch/de/catalog/product/view/id/22352/s/hundenahrung-dentalstoeckchen-1017601/category/54/            2.4
    264                           https://sortiment.lidl.ch/de/catalog/product/view/id/21852/s/waschmittel-0227829/category/54/            3.2
    562                  https://sortiment.lidl.ch/de/catalog/product/view/id/20849/s/schweizer-kartoffeln-0083292/category/54/            3.3
   1012             https://sortiment.lidl.ch/de/catalog/product/view/id/19043/s/cherrystrauchtomaten-lose-0082380/category/54/           10.9
   1048                    https://sortiment.lidl.ch/de/catalog/product/view/id/18360/s/recybag-sammelsack-5113234/category/54/           16.0
   1049                    https://sortiment.lidl.ch/de/catalog/product/view/id/18359/s/recy

In [92]:
# Promotion present but both promotion dates are missing

promotion_present = (
    df["promotion_description"].fillna("").astype(str).str.strip().ne("") |
    df["promotion_price"].notna()
)

dates_missing = (
    df["promotion_valid_from"].fillna("").astype(str).str.strip().eq("") &
    df["promotion_valid_upto"].fillna("").astype(str).str.strip().eq("")
)

mask = promotion_present & dates_missing

issues = df.loc[
    mask,
    [
        "promotion_description",
        "promotion_price",
        "promotion_valid_from",
        "promotion_valid_upto",
        "pdp_url"
    ]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    [
        "row_no",
        "pdp_url",
        "promotion_description",
        "promotion_price",
        "promotion_valid_from",
        "promotion_valid_upto"
    ]
]

print(f"Total issues: {len(issues)}")
print(issues.head(5).to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, promotion_price, promotion_valid_from, promotion_valid_upto]
Index: []


In [120]:
# Check whether price_was is the same as regular_price
# If they are different, the record is reported as an issue.

mask = (
    df['price_was'].notna() &
    df['regular_price'].notna() &
    (df['price_was'].astype(str) != df['regular_price'].astype(str))
)

issues = df.loc[
    mask,
    ['pdp_url', 'price_was', 'regular_price']
].copy()

# Add Excel row number (+2 because row 1 is the header)
issues['row_no'] = issues.index + 2

# Reorder columns
issues = issues[['row_no', 'pdp_url', 'price_was', 'regular_price']]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

KeyError: 'price_was'

In [90]:
# Convert both columns to numeric values
price_was = pd.to_numeric(df["price_was"], errors="coerce")
regular_price = pd.to_numeric(df["regular_price"], errors="coerce")

# Find rows where price_was and regular_price are NOT the same
mask = (
    price_was.notna() &
    regular_price.notna() &
    (price_was != regular_price)
)

issues = df.loc[
    mask,
    ["pdp_url", "price_was", "regular_price"]
].copy()

# Excel row number
issues["row_no"] = issues.index + 2

# Arrange columns
issues = issues[
    ["row_no", "pdp_url", "price_was", "regular_price"]
]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

KeyError: 'price_was'

In [122]:
import pandas as pd
import re

# Extract the unit from price_per_unit
df["price_unit"] = (
    df["price_per_unit"]
    .astype("string")
    .str.extract(
        r'(?:per\s+\d+(?:[.,]\d+)?\s+)?([a-zA-Z]+)',
        flags=re.IGNORECASE,
        expand=False
    )
)

# Normalize both units
df["grammage_unit_clean"] = (
    df["grammage_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["price_unit_clean"] = (
    df["price_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Find rows where units do NOT match
mismatch = df[
    df["grammage_unit_clean"].notna() &
    df["price_unit_clean"].notna() &
    (df["grammage_unit_clean"] != df["price_unit_clean"])
]

# Display relevant columns
print(mismatch[
    ["grammage_unit", "price_per_unit", "price_unit", "pdp_url"]
].to_string(index=False))

print("Total unit mismatch:", len(mismatch))

Empty DataFrame
Columns: [grammage_unit, price_per_unit, price_unit, pdp_url]
Index: []
Total unit mismatch: 0


In [89]:
import pandas as pd
import re

# Extract the unit from price_per_unit
df["price_unit"] = (
    df["price_per_unit"]
    .astype("string")
    .str.extract(
        r'(?:per\s+\d+(?:[.,]\d+)?\s+)?([a-zA-Z]+)',
        flags=re.IGNORECASE,
        expand=False
    )
)

# Normalize both units
df["grammage_unit_clean"] = (
    df["grammage_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["price_unit_clean"] = (
    df["price_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Find rows where units do NOT match
mismatch = df[
    df["grammage_unit_clean"].notna() &
    df["price_unit_clean"].notna() &
    (df["grammage_unit_clean"] != df["price_unit_clean"])
].copy()

# Add row number (+2 because row 1 is the header)
mismatch["row_no"] = mismatch.index + 2

# Display relevant columns
result = mismatch[
    ["row_no", "grammage_unit", "price_per_unit", "price_unit", "pdp_url"]
]

print(result.to_string(index=False))

print(f"\nTotal unit mismatch: {len(result)}")

 row_no grammage_unit     price_per_unit price_unit                                                                                                                            pdp_url
      2         stück        pro 1 stück        pro                                        https://sortiment.lidl.ch/de/catalog/product/view/id/11808/s/zucchetti-0082345/category/54/
      4             g     1Kg = 3.50 CHF         Kg                             https://sortiment.lidl.ch/de/catalog/product/view/id/12790/s/trauben-hell-kernlos-0080505/category/54/
      5         stück        pro 1 stück        pro                             https://sortiment.lidl.ch/de/catalog/product/view/id/13737/s/schweizer-aepfel-rot-0080220/category/54/
      8            kg           pro 1 kg        pro                                https://sortiment.lidl.ch/de/catalog/product/view/id/15344/s/schweizer-rueebli-0082761/category/54/
      9             g          pro 100 g        pro                    https://sortim

In [88]:
# Convert grammage_quantity to numeric
quantity = pd.to_numeric(df["grammage_quantity"], errors="coerce")

# Clean grammage unit
unit = (
    df["grammage_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Find quantity values below 1 but unit is still gram
mask = (
    quantity.notna() &
    (quantity < 1) &
    (unit == "g")
)

issues = df.loc[
    mask,
    ["grammage_quantity", "grammage_unit", "pdp_url"]
].copy()

# Excel row number
issues["row_no"] = issues.index + 2

# Arrange columns
issues = issues[
    ["row_no", "pdp_url", "grammage_quantity", "grammage_unit"]
]

print(f"Total grammage conversion issues: {len(issues)}")
print(issues.to_string(index=False))

Total grammage conversion issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, grammage_quantity, grammage_unit]
Index: []


In [ ]:
import pandas as pd

quantity = pd.to_numeric(
    df["grammage_quantity"],
    errors="coerce"
)

unit = (
    df["grammage_unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Suspicious quantity/unit combinations
mask = (
    ((unit == "g") & (quantity < 1)) |
    ((unit == "kg") & (quantity >= 1000)) |
    ((unit == "ml") & (quantity < 1)) |
    ((unit == "l") & (quantity >= 1000)) |
    ((unit == "mg") & (quantity >= 1000))
)

issues = df.loc[
    mask,
    ["grammage_quantity", "grammage_unit", "pdp_url"]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    ["row_no", "pdp_url", "grammage_quantity", "grammage_unit"]
]

print(f"Total suspicious quantity/unit issues: {len(issues)}")
print(issues.to_string(index=False))

Total suspicious quantity/unit issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, grammage_quantity, grammage_unit]
Index: []


In [ ]:
import pandas as pd

# Words that should be treated as invalid/missing text
invalid_words = [
    "null",
    "none",
    "empty",
    "n/a",
    "na",
    "nan",
    "nil",
    "missing",
    "undefined"
]

issues = []

for col in df.columns:
    # Check only object/string columns
    if df[col].dtype == "object":
        
        mask = (
            df[col]
            .astype("string")
            .str.strip()
            .str.lower()
            .isin(invalid_words)
        )

        if mask.any():
            temp = df.loc[mask, [col, "pdp_url"]].copy()
            temp["row_no"] = temp.index + 2
            temp["column"] = col
            temp["issue_value"] = temp[col]

            issues.append(
                temp[["row_no", "pdp_url", "column", "issue_value"]]
            )

if issues:
    result = pd.concat(issues, ignore_index=True)

    print(f"Total invalid missing-value text records: {len(result)}")
    print(result.to_string(index=False))
else:
    print("No 'null', 'none', 'empty', 'N/A', etc. text values found.")

No 'null', 'none', 'empty', 'N/A', etc. text values found.


In [127]:
issues = df[
    (
        df['promotion_price'].isna() &
        df['promotion_type'].isna() &
        df['percentage_discount'].isna()
    )
    &
    (
        df['regular_price'].isna() |
        df['selling_price'].isna()
    )
].copy()

issues.insert(0, 'row_no', issues.index + 2)

issues[['row_no', 'pdp_url', 'regular_price', 'selling_price',
        'promotion_price', 'promotion_type', 'percentage_discount']]

KeyError: 'promotion_type'

In [128]:
import pandas as pd

# Convert prices to numeric
regular_price = pd.to_numeric(df["regular_price"], errors="coerce")
selling_price = pd.to_numeric(df["selling_price"], errors="coerce")
promotion_price = pd.to_numeric(df["promotion_price"], errors="coerce")

# Check where regular_price exists but promotion_price != selling_price
mask = (
    regular_price.notna() &
    selling_price.notna() &
    (
        promotion_price.isna() |
        (promotion_price != selling_price)
    )
)

issues = df.loc[
    mask,
    [
        "pdp_url",
        "regular_price",
        "selling_price",
        "promotion_price"
    ]
].copy()

# Add row number
issues["row_no"] = issues.index + 2

# Arrange columns
issues = issues[
    [
        "row_no",
        "pdp_url",
        "regular_price",
        "selling_price",
        "promotion_price"
    ]
]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

Total issues: 10576
 row_no                                                                                                                                                           pdp_url  regular_price  selling_price  promotion_price
      2                                                                                         https://www.dm.si/p/d/1662948/bugatti-darilni-set-signature-black-2-delni          29.95          29.95              NaN
      3                                                                                          https://www.dm.si/p/d/2972477/tesori-d-oriente-darilni-set-karma-2-delni           8.95           8.95              NaN
      4                                                                                https://www.dm.si/p/d/3166365/ariel-tekoci-detergent-za-pranje-perila-winter-dream          14.45          14.45              NaN
      5                                                                                                   https:

In [85]:
import pandas as pd

# Function to identify non-empty values
def is_present(s):
    return s.notna() & s.astype(str).str.strip().ne('')


# -------------------------------
# 1. Promotion description present
# -------------------------------

promo = is_present(df['promotion_description'])


# -------------------------------
# 2. Check price fields
# -------------------------------

price_cols = [
    'promotion_price',
    'price_was',
    'selling_price',
    'regular_price'
]

price_count = df[price_cols].apply(is_present).sum(axis=1)


# -------------------------------
# RULE 1
# Promotion description present
# but promotion_price != selling_price
# -------------------------------

promotion_price = pd.to_numeric(
    df['promotion_price'], errors='coerce'
)

selling_price = pd.to_numeric(
    df['selling_price'], errors='coerce'
)

rule1 = (
    promo &
    (
        df['promotion_price'].isna() |
        df['selling_price'].isna() |
        (promotion_price != selling_price)
    )
)


# -------------------------------
# RULE 2
# Only ONE price is present
# but promotion description is present
# -------------------------------

rule2 = promo & (price_count == 1)


# -------------------------------
# Combine violations
# -------------------------------

violations = df[rule1 | rule2].copy()


# -------------------------------
# Add row number from index
# -------------------------------

violations.insert(
    0,
    'row_no',
    violations.index
)


# -------------------------------
# Add issue reason

# -------------------------------

violations['issue'] = ''

violations.loc[
    rule1[violations.index],
    'issue'
] += 'Promotion price/selling price mismatch; '

violations.loc[
    rule2[violations.index],
    'issue'
] += 'Only one price present but promotion description populated; '


# -------------------------------
# Final output
# -------------------------------

result = violations[
    [
        'row_no',
        'pdp_url',
        'promotion_description',
        'promotion_price',
        'price_was',
        'selling_price',
        'regular_price',
        'issue'
    ]
]

print("Total violating products:", len(result))

print(result.to_string(index=False))

KeyError: "['price_was'] not in index"

In [84]:
mask = (
    df["price_valid_from"].fillna("").astype(str).str.strip()
    !=
    df["promotion_valid_from"].fillna("").astype(str).str.strip()
)

result = df.loc[mask, ["pdp_url", "price_valid_from", "promotion_valid_from"]].copy()
result["row_no"] = result.index + 2

print("Total mismatches:", len(result))
print(result[["row_no", "pdp_url", "price_valid_from", "promotion_valid_from"]].to_string(index=False))

Total mismatches: 3863
 row_no                                                                                                                               pdp_url price_valid_from promotion_valid_from
      2                                           https://sortiment.lidl.ch/de/catalog/product/view/id/11808/s/zucchetti-0082345/category/54/              NaN                  NaT
      3                                      https://sortiment.lidl.ch/de/catalog/product/view/id/12995/s/pouletfluegeli-0001363/category/54/              NaN                  NaT
      4                                https://sortiment.lidl.ch/de/catalog/product/view/id/12790/s/trauben-hell-kernlos-0080505/category/54/              NaN                  NaT
      5                                https://sortiment.lidl.ch/de/catalog/product/view/id/13737/s/schweizer-aepfel-rot-0080220/category/54/              NaN                  NaT
      6                              https://sortiment.lidl.ch/de/catalog/pro

In [240]:
df.dtypes

unique_id                              object
competitor_name                        object
extraction_date                        object
product_name                           object
grammage_quantity                      object
grammage_unit                          object
regular_price                         float64
selling_price                         float64
price_valid_from                      float64
price_per_unit                        float64
percentage_discount                   float64
promotion_price                       float64
promotion_valid_from                  float64
promotion_valid_upto                  float64
promotion_description                  object
currency                               object
breadcrumb                             object
pdp_url                                object
region                                float64
pack_size                             float64
site_shown_uom                         object
promotion_description_clean    str

In [83]:
mask = (
    df["percentage_discount"]
    .fillna("")
    .astype(str)
    .str.contains(r"[%\-]", regex=True)
)

result = df.loc[
    mask,
    ["pdp_url", "percentage_discount"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[["row_no", "pdp_url", "percentage_discount"]].to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, percentage_discount]
Index: []


In [85]:
mask = (
    df["regular_price"].fillna("").astype(str).str.strip()
    !=
    df["price_was"].fillna("").astype(str).str.strip()
)

result = df.loc[
    mask,
    ["pdp_url", "regular_price", "price_was"]
].copy()

result["row_no"] = result.index + 2

print("Total mismatches:", len(result))
print(result[["row_no", "pdp_url", "regular_price", "price_was"]].to_string(index=False))

KeyError: 'price_was'

In [243]:
df[df.image_url_1.isna()].pdp_url

AttributeError: 'DataFrame' object has no attribute 'image_url_1'

In [83]:
# Promotion description and promotion price are present,
# but promotion dates are missing

mask = (
    df["promotion_description"].fillna("").astype(str).str.strip().ne("")
    & df["promotion_price"].fillna("").astype(str).str.strip().ne("")
    & (
        df["promotion_valid_from"].fillna("").astype(str).str.strip().eq("")
        | df["promotion_valid_upto"].fillna("").astype(str).str.strip().eq("")
    )
)

result = df.loc[
    mask,
    ["pdp_url", "promotion_description", "promotion_price",
     "pr"
     "omotion_valid_from", "promotion_valid_upto"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[[
    "row_no", "pdp_url", "promotion_description", "promotion_price",
    "promotion_valid_from", "promotion_valid_upto"
]].tail(10).to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, promotion_price, promotion_valid_from, promotion_valid_upto]
Index: []


In [84]:
# Promotion description and promotion price are present,
# but BOTH promotion dates are missing

mask = (
    df["promotion_description"].fillna("").astype(str).str.strip().ne("")
    & df["promotion_price"].fillna("").astype(str).str.strip().ne("")
    & (
        df["promotion_valid_from"].fillna("").astype(str).str.strip().eq("")
        & df["promotion_valid_upto"].fillna("").astype(str).str.strip().eq("")
    )
)

result = df.loc[
    mask,
    ["pdp_url", "promotion_description", "promotion_price",
     "promotion_valid_from", "promotion_valid_upto"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(
    result[[
        "row_no", "pdp_url", "promotion_description", "promotion_price",
        "promotion_valid_from", "promotion_valid_upto"
    ]].tail(10).to_string(index=False)
)

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, promotion_price, promotion_valid_from, promotion_valid_upto]
Index: []


In [82]:
# Check regular_price and selling_price for values
# that are not in exactly two-decimal format

for col in ["regular_price", "selling_price"]:
    price = df[col].fillna("").astype(str).str.strip()

    mask = price.ne("") & ~price.str.fullmatch(r"\d+\.\d{2}", na=False)

    result = df.loc[mask, ["pdp_url", col]].copy()
    result["row_no"] = result.index + 2

    print(f"\nInvalid {col}: {len(result)}")
    print(result[["row_no", "pdp_url", col]].head(5).to_string(index=False))


Invalid regular_price: 302
 row_no                                                                   pdp_url  regular_price
     78 https://shop.billa.at/produkte/rauch-weizenmehl-extra-fuer-kuchen-0015865            1.8
     81                 https://shop.billa.at/produkte/oelz-mehrkorntoast-0014336            3.8
    160              https://shop.billa.at/produkte/hennessy-very-special-0011379           47.9
    244         https://shop.billa.at/produkte/stangl-fadennudeln-mit-ei-00196840            3.8
    247          https://shop.billa.at/produkte/stangl-bandnudeln-mit-ei-00196865            3.8

Invalid selling_price: 311
 row_no                                                                   pdp_url  selling_price
     78 https://shop.billa.at/produkte/rauch-weizenmehl-extra-fuer-kuchen-0015865            1.8
     81                 https://shop.billa.at/produkte/oelz-mehrkorntoast-0014336            3.8
    160              https://shop.billa.at/produkte/hennessy-very-speci

In [85]:
import pandas as pd

from_date = pd.to_datetime(
    df["promotion_valid_from"],
    errors="coerce",
    dayfirst=True
)

upto_date = pd.to_datetime(
    df["promotion_valid_upto"],
    errors="coerce",
    dayfirst=True
)

mask = (
    from_date.notna()
    & upto_date.notna()
    & (from_date > upto_date)
)

result = df.loc[
    mask,
    ["pdp_url", "promotion_valid_from", "promotion_valid_upto"]
].copy()

result["row_no"] = result.index + 2

print("Total date errors:", len(result))
print(
    result[
        ["row_no", "pdp_url", "promotion_valid_from", "promotion_valid_upto"]
    ].to_string(index=False)
)

Total date errors: 1759
 row_no                                                                                                                    pdp_url promotion_valid_from promotion_valid_upto
     30                                                           https://online.spar.si/p/bio-motovilec-spar-naturpur-100g-498215           2026-10-28           2026-04-11
     34                                                                    https://online.spar.si/p/solata-ledenka-spar-kos-659527           2026-10-28           2026-04-11
     81                                                          https://online.spar.si/p/gobe-bukov-ostrigar-pakirano-250g-121564           2026-10-28           2026-04-11
    104                                                             https://online.spar.si/p/kislo-zelje-v-vrecki-spar-500g-543001           2026-10-28           2026-04-11
    117                                                                        https://online.spar.si/p/cvetaca

In [84]:
# Promotion price exists, but regular_price != price_was

mask = (
    df["promotion_price_num"].notna()
    & df["regular_price_num"].notna()
    & df["price_was_num"].notna()
    & df["regular_price_num"].ne(df["price_was_num"])
)

result = df.loc[
    mask,
    ["unique_id", "pdp_url", "regular_price", "price_was", "promotion_price"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[
    ["row_no", "unique_id", "pdp_url", "regular_price", "price_was", "promotion_price"]
].to_string(index=False))

KeyError: 'promotion_price_num'

In [ ]:
mask = (
    df["regular_price_num"].notna()
    & df["selling_price_num"].notna()
    & df["regular_price_num"].lt(df["selling_price_num"])
)

result = df.loc[
    mask,
    ["unique_id", "pdp_url", "regular_price", "selling_price"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[
    ["row_no", "unique_id", "pdp_url", "regular_price", "selling_price"]
].to_string(index=False))

KeyError: 'regular_price_num'

In [ ]:
mask = (
    df["promotion_price_num"].notna()
    & df["regular_price_num"].notna()
    & df["promotion_price_num"].gt(df["regular_price_num"])
)

result = df.loc[
    mask,
    ["unique_id", "pdp_url", "regular_price", "promotion_price"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(result[
    ["row_no", "unique_id", "pdp_url", "regular_price", "promotion_price"]
].to_string(index=False))

KeyError: 'promotion_price_num'

In [48]:
# Check invalid promotion dates
# promotion_valid_from > promotion_valid_upto

df["promotion_valid_from"] = pd.to_datetime(
    df["promotion_valid_from"], errors="coerce"
)

df["promotion_valid_upto"] = pd.to_datetime(
    df["promotion_valid_upto"], errors="coerce"
)

mask = (
    df["promotion_valid_from"].notna()
    & df["promotion_valid_upto"].notna()
    & (df["promotion_valid_from"] > df["promotion_valid_upto"])
)

result = df.loc[
    mask,
    ["unique_id", "pdp_url", "promotion_valid_from", "promotion_valid_upto"]
].copy()

result["row_no"] = result.index + 2

print("Total invalid promotion date records:", len(result))
print(
    result[
        ["row_no", "unique_id", "pdp_url", "promotion_valid_from", "promotion_valid_upto"]
    ].to_string(index=False)
)

Total invalid promotion date records: 0
Empty DataFrame
Columns: [row_no, unique_id, pdp_url, promotion_valid_from, promotion_valid_upto]
Index: []


In [47]:
import pandas as pd

# Convert price columns to numeric
price_cols = ["regular_price", "selling_price", "promotion_price"]

for col in price_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Regular price is lower than selling price or promotion price
mask = (
    (df["selling_price"].notna() & (df["regular_price"] < df["selling_price"]))
    | (df["promotion_price"].notna() & (df["regular_price"] < df["promotion_price"]))
)

result = df.loc[
    mask,
    ["unique_id", "pdp_url", "regular_price", "selling_price", "promotion_price"]
].copy()

result["row_no"] = result.index + 2

print("Total issues:", len(result))
print(
    result[
        ["row_no", "unique_id", "pdp_url", "regular_price", "selling_price", "promotion_price"]
    ].to_string(index=False)
)

Total issues: 0
Empty DataFrame
Columns: [row_no, unique_id, pdp_url, regular_price, selling_price, promotion_price]
Index: []


In [46]:
import pandas as pd

# Check where all three prices are equal
price_cols = ["regular_price", "selling_price", "promotion_price"]

prices = df[price_cols].apply(pd.to_numeric, errors="coerce")

mask = (
    prices.notna().all(axis=1)
    & prices["regular_price"].eq(prices["selling_price"])
    & prices["selling_price"].eq(prices["promotion_price"])
)

result = df.loc[mask, ["pdp_url"] + price_cols].copy()
result.insert(0, "row_no", result.index + 2)

print("Total records with all prices equal:", len(result))

print(result.to_string(index=False))

Total records with all prices equal: 0
Empty DataFrame
Columns: [row_no, pdp_url, regular_price, selling_price, promotion_price]
Index: []


In [ ]:
df.grammage_quantity.value_counts()